# Training GRU BISINDO - Colab **v3** (multi-select schema/model/suite)

Versi lanjutan dari `train_bisindo_gru_colab_v2.ipynb`. Notebook ini tetap mandiri untuk Colab: modul training dari repo di-*embed* ke `/content/src`, lalu dipakai langsung oleh cell training.

## Yang bisa dipilih (checkbox)
- **Model dasar**: khukuh / adi / hybrid / biattn / convfront / tcn / transformer.
- **Schema**: 8 schema aktual, termasuk face-reference baru.
- **Mode data**: original dan/atau dengan augmentasi; mode augmentasi otomatis memakai variant `_dengan_augmentasi`.
- **Suite**: main / chunk10 / threshold / boosted.

Notebook akan melatih semua kombinasi silang dari checkbox: **model x schema x mode data x suite**. Default awal dibuat aman: `adi + smart180 + original + main` saja.

## Catatan penting
- `chunk10` dan `threshold` melatih expert per grup, jadi bisa lama. Gunakan `SUITE_EPOCHS` untuk membatasi epoch expert.
- `boosted` butuh checkpoint main; notebook akan memastikan main checkpoint ada sebelum suite expert dijalankan.
- `OVERWRITE_EXISTING` default `False`: checkpoint yang sudah ada akan di-skip kecuali checkbox overwrite dicentang.
- Runtime Colab: pilih **Runtime -> Change runtime type -> GPU** sebelum training.


In [ ]:
#@title 1. Cek environment & GPU
import sys, platform
print("Python:", sys.version.split()[0], "|", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("⚠️  GPU OFF — Runtime > Change runtime type > GPU, lalu Restart")
except Exception as e:
    print("torch belum siap:", e)
import numpy, pandas, pyarrow, tqdm, sklearn
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| pyarrow", pyarrow.__version__,
      "| tqdm", tqdm.__version__, "| sklearn", sklearn.__version__)


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Tulis source modul (verbatim dari repo) ke /content/src
import base64, os, sys

SRC_ROOT = "/content/src"
EMBEDDED = {'smart_extract/__init__.py': 'IiIiU21hcnQgRXh0cmFjdCBWOCBwYWNrYWdlLiIiIgoK', 'smart_extract/contract.py': 'IiIiU2hhcmVkIFNtYXJ0IEV4dHJhY3QgVjggYmVzdC1tb2RlIGZlYXR1cmUgY29udHJhY3QuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGFyZ3BhcnNlIGltcG9ydCBOYW1lc3BhY2UKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBJdGVyYWJsZQoKaW1wb3J0IG51bXB5IGFzIG5wCgpGRUFUVVJFX1NDSEVNQSA9ICJiaXNpbmRvX3NtYXJ0X3Y4X2J0al9nbG9iYWxfbG9jYWxfMTgwX2Jlc3QiCkZFQVRVUkVfTU9ERSA9ICJidGpfZ2xvYmFsX2xvY2FsIgpGRUFUVVJFX0RJTSA9IDE4MApUQVJHRVRfRlBTID0gMTAuMApFWFRSQUNUX1BST0ZJTEUgPSAic21hcnRfdjhfYmVzdCIKClNMSUNFX1NIT1VMREVSUyA9IHNsaWNlKDAsIDYpClNMSUNFX0xFRlRfR0xPQkFMID0gc2xpY2UoNiwgMzkpClNMSUNFX1JJR0hUX0dMT0JBTCA9IHNsaWNlKDM5LCA3MikKU0xJQ0VfTEVGVF9MT0NBTCA9IHNsaWNlKDcyLCAxMDUpClNMSUNFX1JJR0hUX0xPQ0FMID0gc2xpY2UoMTA1LCAxMzgpClNMSUNFX0xFRlRfQU5HTEVTID0gc2xpY2UoMTM4LCAxNTQpClNMSUNFX1JJR0hUX0FOR0xFUyA9IHNsaWNlKDE1NCwgMTcwKQpTTElDRV9NRVRBID0gc2xpY2UoMTcwLCAxODApCgpJRFhfTUVUQV9MRUZUX1BSRVNFTlQgPSAwCklEWF9NRVRBX1JJR0hUX1BSRVNFTlQgPSAxCklEWF9NRVRBX0xFRlRfREVURUNURUQgPSAyCklEWF9NRVRBX1JJR0hUX0RFVEVDVEVEID0gMwpJRFhfTUVUQV9MRUZUX0hFTEQgPSA0CklEWF9NRVRBX1JJR0hUX0hFTEQgPSA1CklEWF9NRVRBX1NIT1VMREVSX09LID0gNgpJRFhfTUVUQV9TSE9VTERFUl9TQ0FMRSA9IDcKSURYX01FVEFfTEVGVF9TQ09SRSA9IDgKSURYX01FVEFfUklHSFRfU0NPUkUgPSA5CgpCRVNUX0ZBTExCQUNLX1ZBUklBTlRTID0gImF1dG8sY2xhaGVfc2hhcnAsZ2FtbWFfYnJpZ2h0LHNoYXJwLGRlbm9pc2VfY2xhaGVfc2hhcnAsbm9uZSIKCkJFU1RfRVhUUkFDVF9TRVRUSU5HUyA9IHsKICAgICJmZWF0dXJlX21vZGUiOiBGRUFUVVJFX01PREUsCiAgICAidGFyZ2V0X2ZwcyI6IFRBUkdFVF9GUFMsCiAgICAid2lkdGgiOiA2NDAsCiAgICAiaGVpZ2h0IjogNDgwLAogICAgImNlbnRlcl9jcm9wIjogMS4wLAogICAgInByb2Nfd2lkdGgiOiAzODQsCiAgICAic2hvdWxkZXJfYmFja2VuZCI6ICJtcC1wb3NlIiwKICAgICJwb3NlX2V2ZXJ5IjogMywKICAgICJwb3NlX3Byb2Nfd2lkdGgiOiAyNTYsCiAgICAiaGFuZF9tb2RlbF9jb21wbGV4aXR5IjogMCwKICAgICJwb3NlX21vZGVsX2NvbXBsZXhpdHkiOiAwLAogICAgImRldF9jb25mIjogMC40MCwKICAgICJ0cmFja19jb25mIjogMC40NSwKICAgICJzbW9vdGhfYWxwaGEiOiAwLjc4LAogICAgInNob3VsZGVyX3Ntb290aF9hbHBoYSI6IDAuMzUsCiAgICAiaG9sZF9mcmFtZXMiOiA1LAogICAgInNtYXJ0X21vZGUiOiAiYmVzdCIsCiAgICAic2VhcmNoX3JhZGl1cyI6IDIsCiAgICAiZW5oYW5jZSI6ICJhdXRvIiwKICAgICJmYWxsYmFja192YXJpYW50cyI6IEJFU1RfRkFMTEJBQ0tfVkFSSUFOVFMsCiAgICAiZ2lmX3dpZHRoIjogNDIwLAp9CgoKZGVmIG1ha2VfYmVzdF9hcmdzKCoqb3ZlcnJpZGVzKSAtPiBOYW1lc3BhY2U6CiAgICAiIiJSZXR1cm4gYW4gYXJncGFyc2UtY29tcGF0aWJsZSBuYW1lc3BhY2UgZm9yIHRoZSBiZXN0IGV4dHJhY3QgcHJvZmlsZS4iIiIKICAgIHZhbHVlcyA9IHsKICAgICAgICAidmlkZW8iOiBOb25lLAogICAgICAgICJiYXRjaF9kaXIiOiBOb25lLAogICAgICAgICJvdXRfZGlyIjogTm9uZSwKICAgICAgICAic2F2ZV9naWYiOiBUcnVlLAogICAgICAgICJub19naWYiOiBGYWxzZSwKICAgICAgICAic2F2ZV9tcDQiOiBGYWxzZSwKICAgICAgICAic2tlbGV0b25fYmciOiAiYmxhY2siLAogICAgICAgICJxdWlldCI6IEZhbHNlLAogICAgICAgICJtaXJyb3JfaW5wdXQiOiBGYWxzZSwKICAgICAgICAibm9fbWlycm9yX2hhbmRlZG5lc3MiOiBGYWxzZSwKICAgICAgICAid2Vha19oYW5kX3RocmVzaG9sZCI6IDEuNSwKICAgIH0KICAgIHZhbHVlcy51cGRhdGUoQkVTVF9FWFRSQUNUX1NFVFRJTkdTKQogICAgdmFsdWVzLnVwZGF0ZShvdmVycmlkZXMpCiAgICByZXR1cm4gTmFtZXNwYWNlKCoqdmFsdWVzKQoKCmRlZiBwYXJzZV9mZWF0dXJlX3ZhbHVlKHZhbHVlKSAtPiBucC5uZGFycmF5OgogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKToKICAgICAgICByZXR1cm4gbnAuZnJvbXN0cmluZyh2YWx1ZSwgc2VwPSIsIiwgZHR5cGU9bnAuZmxvYXQzMikKICAgIHJldHVybiBucC5hc2FycmF5KHZhbHVlLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQoKCmRlZiBmb3JtYXRfZmVhdHVyZV92YWx1ZShmZWF0dXJlczogSXRlcmFibGVbZmxvYXRdKSAtPiBzdHI6CiAgICByZXR1cm4gIiwiLmpvaW4obWFwKHN0ciwgbnAuYXNhcnJheShmZWF0dXJlcywgZHR5cGU9bnAuZmxvYXQzMikucmVzaGFwZSgtMSkudG9saXN0KCkpKQoKCmRlZiBlbnN1cmVfZmVhdHVyZV9kaW0oc2VxdWVuY2UsIGV4cGVjdGVkX2RpbTogaW50ID0gRkVBVFVSRV9ESU0pIC0+IG5wLm5kYXJyYXk6CiAgICBhcnIgPSBucC5hc2FycmF5KHNlcXVlbmNlLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgYXJyLm5kaW0gPT0gMToKICAgICAgICBhcnIgPSBhcnIucmVzaGFwZSgxLCAtMSkKICAgIGlmIGFyci5uZGltICE9IDIgb3IgYXJyLnNoYXBlWzFdICE9IGludChleHBlY3RlZF9kaW0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBmZWF0dXJlIHNoYXBlIChULCB7ZXhwZWN0ZWRfZGltfSksIGdvdCB7YXJyLnNoYXBlfSIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoYXJyKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGZWF0dXJlIHNlcXVlbmNlIGNvbnRhaW5zIG5vbi1maW5pdGUgdmFsdWVzIikKICAgIHJldHVybiBhcnIuYXN0eXBlKG5wLmZsb2F0MzIsIGNvcHk9RmFsc2UpCgoKZGVmIGZpbHRlcl9jdXJyZW50X2ZlYXR1cmVfcm93cyhkZik6CiAgICBpZiAiZmVhdHVyZV92ZXJzaW9uIiBub3QgaW4gZGYuY29sdW1uczoKICAgICAgICByZXR1cm4gZGYuaWxvY1swOjBdLmNvcHkoKQogICAgaWYgImZlYXR1cmVfZGltIiBpbiBkZi5jb2x1bW5zOgogICAgICAgIGRpbXMgPSBkZlsiZmVhdHVyZV9kaW0iXQogICAgICAgIHRyeToKICAgICAgICAgICAgZGltcyA9IGRpbXMuYXN0eXBlKGludCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBkZWYgX2RpbSh2YWx1ZSk6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGludChmbG9hdCh2YWx1ZSkpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHJldHVybiAtMQogICAgICAgICAgICBkaW1zID0gZGltcy5hcHBseShfZGltKQogICAgICAgIHJldHVybiBkZlsoZGZbImZlYXR1cmVfdmVyc2lvbiJdID09IEZFQVRVUkVfU0NIRU1BKSAmIChkaW1zID09IEZFQVRVUkVfRElNKV0uY29weSgpCiAgICByZXR1cm4gZGZbZGZbImZlYXR1cmVfdmVyc2lvbiJdID09IEZFQVRVUkVfU0NIRU1BXS5jb3B5KCkKCgpkZWYgc2FtcGxlX2dpZl9wYXRocyh2b2NhYjogc3RyLCB2aWRlb19pZDogc3RyLCByb290X2Rpcjogc3RyIHwgUGF0aCwgbW9kZXM6IEl0ZXJhYmxlW3N0cl0gPSAoIm92ZXJsYXkiLCAic2tlbGV0b24iKSkgLT4gZGljdFtzdHIsIHN0cl06CiAgICBzYWZlX3ZpZGVvX2lkID0gIiIuam9pbihjIGlmIGMuaXNhbG51bSgpIG9yIGMgaW4gIi5fLSIgZWxzZSAiXyIgZm9yIGMgaW4gc3RyKHZpZGVvX2lkKSkKICAgIGJhc2UgPSBQYXRoKHJvb3RfZGlyKSAvICJhc3NldHMiIC8gImdpZnMiIC8gInNhbXBsZXMiIC8gc3RyKHZvY2FiKQogICAgcmV0dXJuIHttb2RlOiBzdHIoYmFzZSAvIGYie3NhZmVfdmlkZW9faWR9X3ttb2RlfS5naWYiKSBmb3IgbW9kZSBpbiBtb2Rlc30KCgpkZWYgbW90aW9uX3Njb3JlKHByZXY6IG5wLm5kYXJyYXkgfCBOb25lLCBjdXJyOiBucC5uZGFycmF5IHwgTm9uZSkgLT4gdHVwbGVbZmxvYXQsIGJvb2xdOgogICAgaWYgY3VyciBpcyBOb25lOgogICAgICAgIHJldHVybiAwLjAsIEZhbHNlCiAgICBjdXJyID0gbnAuYXNhcnJheShjdXJyLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgaWYgY3Vyci5zaGFwZVswXSA8IEZFQVRVUkVfRElNOgogICAgICAgIHJldHVybiAwLjAsIEZhbHNlCiAgICBtZXRhID0gY3VycltTTElDRV9NRVRBXQogICAgdmlzaWJsZSA9IGJvb2wobWV0YVtJRFhfTUVUQV9MRUZUX1BSRVNFTlRdID49IDAuNSBvciBtZXRhW0lEWF9NRVRBX1JJR0hUX1BSRVNFTlRdID49IDAuNSkKICAgIGlmIHByZXYgaXMgTm9uZToKICAgICAgICByZXR1cm4gMC4wLCB2aXNpYmxlCiAgICBwcmV2ID0gbnAuYXNhcnJheShwcmV2LCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgaWYgcHJldi5zaGFwZVswXSA8IEZFQVRVUkVfRElNOgogICAgICAgIHJldHVybiAwLjAsIHZpc2libGUKICAgIGNodW5rcyA9IFsKICAgICAgICAoU0xJQ0VfTEVGVF9HTE9CQUwsIElEWF9NRVRBX0xFRlRfUFJFU0VOVCksCiAgICAgICAgKFNMSUNFX1JJR0hUX0dMT0JBTCwgSURYX01FVEFfUklHSFRfUFJFU0VOVCksCiAgICAgICAgKFNMSUNFX0xFRlRfTE9DQUwsIElEWF9NRVRBX0xFRlRfUFJFU0VOVCksCiAgICAgICAgKFNMSUNFX1JJR0hUX0xPQ0FMLCBJRFhfTUVUQV9SSUdIVF9QUkVTRU5UKSwKICAgIF0KICAgIHNjb3JlcyA9IFtdCiAgICBmb3Igc2wsIG1ldGFfaWR4IGluIGNodW5rczoKICAgICAgICBpZiBtZXRhW21ldGFfaWR4XSA+PSAwLjU6CiAgICAgICAgICAgIHNjb3Jlcy5hcHBlbmQoZmxvYXQobnAubGluYWxnLm5vcm0oY3VycltzbF0gLSBwcmV2W3NsXSkpKQogICAgcmV0dXJuIChmbG9hdChtYXgoc2NvcmVzKSkgaWYgc2NvcmVzIGVsc2UgMC4wKSwgdmlzaWJsZQo=', 'smart_extract/live_bisindo_mp_real_shoulder_v6.py': '...', 'feature_schemas.py': '...', 'face_reference.py': 'IiIiRmFjZS1yZWZlcmVuY2UgZmVhdHVyZSBibG9ja3MgZm9yIHNtYXJ0MTgwLWJhc2VkIEJJU0lORE8gc2NoZW1hcy4KClRoZSBmYWNlIGhlcmUgaXMgYSAqcG9zaXRpb24gcmVmZXJlbmNlIGZvciB0aGUgaGFuZHMqLCBub3QgYSBtb3Rpb24gc291cmNlLgpFdmVyeSBmYWNlL2hhbmQgcG9pbnQgaXMgbm9ybWFsaXNlZCBpbiB0aGUgc2FtZSBzaG91bGRlciBmcmFtZSB1c2VkIGJ5IHNtYXJ0MTgwCihgYHNob3VsZGVyX2FuY2hvcl9zY2FsZWBgKSwgc28gYSBmYWNlLXJlZiB2ZWN0b3IgaXMganVzdCBgYHNtYXJ0MTgwWzA6MTgwXWBgCmZvbGxvd2VkIGJ5IG9uZSBleHRyYSBibG9jay4gYGBidWlsZF9mYWNlX3NjaGVtYV92ZWN0b3JgYCBpcyB0aGUgc2luZ2xlIGVudHJ5CnBvaW50IHNoYXJlZCBieSB0aGUgb2ZmbGluZSBjb252ZXJ0ZXIgYW5kIHRoZSBsaXZlIGV4dHJhY3Rvciwgd2hpY2ggZ2l2ZXMKb2ZmbGluZS9saXZlIHBhcml0eSBieSBjb25zdHJ1Y3Rpb24uCgpXaGVuIHRoZSBmYWNlIChvciBhIGhhbmQpIGlzIG1pc3NpbmcgdGhlIGNvcnJlc3BvbmRpbmcgc3ViLWJsb2NrIGlzIHplcm9lZCBhbmQgYQpgYCpfcHJlc2VudGBgIGZsYWcgZHJvcHMgdG8gMCDigJQgbmV2ZXIgYGAoMCAtIGFuY2hvcikvc2NhbGVgYCBsZWFraW5nIGEgZmFrZQpwb3NpdGlvbi4gVmVsb2NpdHkgdGVybXMgYXJlIHBlci1zZWNvbmQgKGBgZGVsdGEgLyBkdGBgKSBzbyBvZmZsaW5lIDEwIGZwcyBhbmQgYQp2YXJpYWJsZSBsaXZlIGZyYW1lLXJhdGUgc3RheSBjb21wYXJhYmxlOyB0aGUgZmlyc3QgdmFsaWQgZnJhbWUgKGFuZCB0aGUgZnJhbWUKYWZ0ZXIgYSBnYXApIHJlcG9ydHMgemVybyB2ZWxvY2l0eS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gc21hcnRfZXh0cmFjdC5saXZlX2Jpc2luZG9fbXBfcmVhbF9zaG91bGRlcl92NiBpbXBvcnQgKAogICAgcGFsbV9jZW50ZXIsCiAgICBzaG91bGRlcl9hbmNob3Jfc2NhbGUsCikKCgojIC0tLSBGYWNlTWVzaCBsYW5kbWFyayBpbmRpY2VzIChhbGwgPCA0NjgsIHZhbGlkIGZvciByZWZpbmUgb24vb2ZmKSAtLS0tLS0tLS0tLQpOT1NFX1RJUCA9IDEKTU9VVEhfVE9QID0gMTMKTU9VVEhfQk9UVE9NID0gMTQKTU9VVEhfTEVGVCA9IDYxCk1PVVRIX1JJR0hUID0gMjkxCkVZRV9MX09VVEVSID0gMzMKRVlFX0xfSU5ORVIgPSAxMzMKRVlFX1JfSU5ORVIgPSAzNjIKRVlFX1JfT1VURVIgPSAyNjMKCiMgOSBmYWNlIHBvaW50cyB1c2VkIGJ5IHRoZSBtb3V0aGR5biBzY2hlbWEgKHJlbGF0aXZlIGJsb2NrKS4KTU9VVEhEWU5fUE9JTlRTID0gKAogICAgTk9TRV9USVAsCiAgICBNT1VUSF9UT1AsCiAgICBNT1VUSF9CT1RUT00sCiAgICBNT1VUSF9MRUZULAogICAgTU9VVEhfUklHSFQsCiAgICBFWUVfTF9PVVRFUiwKICAgIEVZRV9MX0lOTkVSLAogICAgRVlFX1JfSU5ORVIsCiAgICBFWUVfUl9PVVRFUiwKKQpNT1VUSERZTl9QT0lOVF9OQU1FUyA9ICgKICAgICJub3NlIiwKICAgICJtb3V0aF90b3AiLAogICAgIm1vdXRoX2JvdHRvbSIsCiAgICAibW91dGhfbGVmdCIsCiAgICAibW91dGhfcmlnaHQiLAogICAgImV5ZV9sX291dGVyIiwKICAgICJleWVfbF9pbm5lciIsCiAgICAiZXllX3JfaW5uZXIiLAogICAgImV5ZV9yX291dGVyIiwKKQoKU01BUlRfQkFTRV9ESU0gPSAxODAKCiMgU2VsZWN0ZWQgaGFuZCBwb2ludHMgdXNlZCBieSBzbWFydDE4MCBnbG9iYWwgc2xpY2VzIChmb3IgdmVsb2NpdHkgY29sdW1uIG5hbWVzKS4KX1NNQVJUMTgwX1NFTEVDVEVEX1BPSU5UUyA9ICgKICAgICJ3cmlzdCIsCiAgICAicGFsbV9jZW50ZXIiLAogICAgInRodW1iX3RpcCIsCiAgICAiaW5kZXhfbWNwIiwKICAgICJpbmRleF90aXAiLAogICAgIm1pZGRsZV9tY3AiLAogICAgIm1pZGRsZV90aXAiLAogICAgInJpbmdfbWNwIiwKICAgICJyaW5nX3RpcCIsCiAgICAicGlua3lfbWNwIiwKICAgICJwaW5reV90aXAiLAopCiMgc21hcnQxODAgWzY6MzldID0gbGVmdCBnbG9iYWwsIFszOTo3Ml0gPSByaWdodCBnbG9iYWwgKDExIHBvaW50cyB4IDMpLgpfU0xJQ0VfTEVGVF9HTE9CQUwgPSBzbGljZSg2LCAzOSkKX1NMSUNFX1JJR0hUX0dMT0JBTCA9IHNsaWNlKDM5LCA3MikKCiMgRXh0cmEgYmxvY2sgc2l6ZSBhcHBlbmRlZCBhZnRlciBzbWFydDE4MFswOjE4MF0gZm9yIGVhY2ggc2NoZW1hLgpGQUNFX1JFRl9CTE9DS19ESU1TID0gewogICAgInNtYXJ0MTgwX21vdXRoZHluMjE0IjogMzQsCiAgICAic21hcnQxODBfbW91dGhzdGF0MjA2IjogMjYsCiAgICAic21hcnQxODBfaGFuZGZhY2UyMjAiOiA0MCwKICAgICJzbWFydDE4MF9oYW5kZmFjZV92ZWwyODYiOiAxMDYsCn0KRkFDRV9SRUZfU0NIRU1BX0RJTVMgPSB7bmFtZTogU01BUlRfQkFTRV9ESU0gKyBibG9jayBmb3IgbmFtZSwgYmxvY2sgaW4gRkFDRV9SRUZfQkxPQ0tfRElNUy5pdGVtcygpfQpGQUNFX1JFRl9TQ0hFTUFfTkFNRVMgPSB0dXBsZShGQUNFX1JFRl9CTE9DS19ESU1TLmtleXMoKSkKIyBTY2hlbWFzIHRoYXQga2VlcCB0ZW1wb3JhbCBzdGF0ZSBhY3Jvc3MgZnJhbWVzIChtdXN0IHJ1biBzaW5nbGUtd29ya2VyKS4KRkFDRV9SRUZfU1RBVEVGVUxfU0NIRU1BUyA9ICgic21hcnQxODBfbW91dGhkeW4yMTQiLCAic21hcnQxODBfaGFuZGZhY2VfdmVsMjg2IikKCgpkZWYgX3Nob3VsZGVyX3JlZihzaG91bGRlcnMpIC0+IHR1cGxlW25wLm5kYXJyYXksIGZsb2F0XToKICAgIGFuY2hvciwgc2NhbGUsIF8sIF8gPSBzaG91bGRlcl9hbmNob3Jfc2NhbGUobnAuYXNhcnJheShzaG91bGRlcnMsIGR0eXBlPW5wLmZsb2F0MzIpKQogICAgcmV0dXJuIGFuY2hvci5hc3R5cGUobnAuZmxvYXQzMiksIGZsb2F0KHNjYWxlKQoKCmRlZiBfZmFjZV9wb2ludHMoZmFjZV94eXo6IG5wLm5kYXJyYXksIGluZGljZXMpIC0+IG5wLm5kYXJyYXk6CiAgICByZXR1cm4gbnAuc3RhY2soW25wLmFzYXJyYXkoZmFjZV94eXpbaV0sIGR0eXBlPW5wLmZsb2F0MzIpIGZvciBpIGluIGluZGljZXNdKS5hc3R5cGUobnAuZmxvYXQzMikKCgpkZWYgX3JlbChwb2ludDogbnAubmRhcnJheSwgYW5jaG9yOiBucC5uZGFycmF5LCBzY2FsZTogZmxvYXQpIC0+IG5wLm5kYXJyYXk6CiAgICByZXR1cm4gKChucC5hc2FycmF5KHBvaW50LCBkdHlwZT1ucC5mbG9hdDMyKSAtIGFuY2hvcikgLyBzY2FsZSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIF9idWlsZF9tb3V0aGR5bihmYWNlX3h5eiwgYW5jaG9yLCBzY2FsZSwgZmFjZV9wcmVzZW50LCBzdGF0ZSwgZHQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIyNyBmYWNlLXJlbCArIChvcGVubmVzcyx3aWR0aCxhc3BlY3QsZXllX2Rpc3QpICsgKGRfb3Blbm5lc3MsZF9tb3V0aF95KSArIHByZXNlbnQuIiIiCiAgICBibG9jayA9IG5wLnplcm9zKDM0LCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgbm90IGZhY2VfcHJlc2VudDoKICAgICAgICBpZiBzdGF0ZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgc3RhdGUucG9wKCJtb3V0aGR5bl9wcmV2X29wZW5uZXNzIiwgTm9uZSkKICAgICAgICAgICAgc3RhdGUucG9wKCJtb3V0aGR5bl9wcmV2X21vdXRoX3kiLCBOb25lKQogICAgICAgIHJldHVybiBibG9jawoKICAgIHJlbCA9IChfZmFjZV9wb2ludHMoZmFjZV94eXosIE1PVVRIRFlOX1BPSU5UUykgLSBhbmNob3IpIC8gc2NhbGUgICMgKDksIDMpCiAgICBibG9ja1swOjI3XSA9IHJlbC5yZXNoYXBlKC0xKQogICAgdG9wLCBib3R0b20sIG1fbGVmdCwgbV9yaWdodCA9IHJlbFsxXSwgcmVsWzJdLCByZWxbM10sIHJlbFs0XQogICAgZXllX2xlZnQgPSAocmVsWzVdICsgcmVsWzZdKSAqIDAuNQogICAgZXllX3JpZ2h0ID0gKHJlbFs3XSArIHJlbFs4XSkgKiAwLjUKICAgIG9wZW5uZXNzID0gZmxvYXQobnAubGluYWxnLm5vcm0odG9wIC0gYm90dG9tKSkKICAgIHdpZHRoID0gZmxvYXQobnAubGluYWxnLm5vcm0obV9sZWZ0IC0gbV9yaWdodCkpCiAgICBhc3BlY3QgPSBvcGVubmVzcyAvIG1heCh3aWR0aCwgMWUtNCkKICAgIGV5ZV9kaXN0ID0gZmxvYXQobnAubGluYWxnLm5vcm0oZXllX2xlZnQgLSBleWVfcmlnaHQpKQogICAgbW91dGhfY2VudGVyX3kgPSBmbG9hdCgodG9wWzFdICsgYm90dG9tWzFdICsgbV9sZWZ0WzFdICsgbV9yaWdodFsxXSkgLyA0LjApCiAgICBibG9ja1syN10gPSBvcGVubmVzcwogICAgYmxvY2tbMjhdID0gd2lkdGgKICAgIGJsb2NrWzI5XSA9IGFzcGVjdAogICAgYmxvY2tbMzBdID0gZXllX2Rpc3QKCiAgICBkdF9zID0gbWF4KGZsb2F0KGR0KSwgMWUtMykKICAgIHByZXZfb3BlbiA9IHN0YXRlLmdldCgibW91dGhkeW5fcHJldl9vcGVubmVzcyIpIGlmIHN0YXRlIGlzIG5vdCBOb25lIGVsc2UgTm9uZQogICAgcHJldl95ID0gc3RhdGUuZ2V0KCJtb3V0aGR5bl9wcmV2X21vdXRoX3kiKSBpZiBzdGF0ZSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgIGlmIHByZXZfb3BlbiBpcyBub3QgTm9uZToKICAgICAgICBibG9ja1szMV0gPSAob3Blbm5lc3MgLSBmbG9hdChwcmV2X29wZW4pKSAvIGR0X3MKICAgIGlmIHByZXZfeSBpcyBub3QgTm9uZToKICAgICAgICBibG9ja1szMl0gPSAobW91dGhfY2VudGVyX3kgLSBmbG9hdChwcmV2X3kpKSAvIGR0X3MKICAgIGJsb2NrWzMzXSA9IDEuMAogICAgaWYgc3RhdGUgaXMgbm90IE5vbmU6CiAgICAgICAgc3RhdGVbIm1vdXRoZHluX3ByZXZfb3Blbm5lc3MiXSA9IG9wZW5uZXNzCiAgICAgICAgc3RhdGVbIm1vdXRoZHluX3ByZXZfbW91dGhfeSJdID0gbW91dGhfY2VudGVyX3kKICAgIHJldHVybiBibG9jawoKCmRlZiBfYnVpbGRfbW91dGhzdGF0KGZhY2VfeHl6LCBhbmNob3IsIHNjYWxlLCBmYWNlX3ByZXNlbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJub3NlKDMpICsgbW91dGggbGluZSAobGVmdCxyaWdodCxjZW50ZXIgPSA5KSArIDQgZXllIGNvcm5lcnMoMTIpICsgZXllX2Rpc3QgKyBwcmVzZW50LiIiIgogICAgYmxvY2sgPSBucC56ZXJvcygyNiwgZHR5cGU9bnAuZmxvYXQzMikKICAgIGlmIG5vdCBmYWNlX3ByZXNlbnQ6CiAgICAgICAgcmV0dXJuIGJsb2NrCgogICAgbm9zZSA9IF9yZWwoZmFjZV94eXpbTk9TRV9USVBdLCBhbmNob3IsIHNjYWxlKQogICAgbV9sZWZ0ID0gX3JlbChmYWNlX3h5eltNT1VUSF9MRUZUXSwgYW5jaG9yLCBzY2FsZSkKICAgIG1fcmlnaHQgPSBfcmVsKGZhY2VfeHl6W01PVVRIX1JJR0hUXSwgYW5jaG9yLCBzY2FsZSkKICAgIG1vdXRoX2NlbnRlcl9yYXcgPSAoCiAgICAgICAgbnAuYXNhcnJheShmYWNlX3h5eltNT1VUSF9UT1BdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICsgbnAuYXNhcnJheShmYWNlX3h5eltNT1VUSF9CT1RUT01dLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICsgbnAuYXNhcnJheShmYWNlX3h5eltNT1VUSF9MRUZUXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICArIG5wLmFzYXJyYXkoZmFjZV94eXpbTU9VVEhfUklHSFRdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgKSAvIDQuMAogICAgbW91dGhfY2VudGVyID0gX3JlbChtb3V0aF9jZW50ZXJfcmF3LCBhbmNob3IsIHNjYWxlKQogICAgZV9sbyA9IF9yZWwoZmFjZV94eXpbRVlFX0xfT1VURVJdLCBhbmNob3IsIHNjYWxlKQogICAgZV9saSA9IF9yZWwoZmFjZV94eXpbRVlFX0xfSU5ORVJdLCBhbmNob3IsIHNjYWxlKQogICAgZV9yaSA9IF9yZWwoZmFjZV94eXpbRVlFX1JfSU5ORVJdLCBhbmNob3IsIHNjYWxlKQogICAgZV9ybyA9IF9yZWwoZmFjZV94eXpbRVlFX1JfT1VURVJdLCBhbmNob3IsIHNjYWxlKQogICAgZXllX2xlZnQgPSAoZV9sbyArIGVfbGkpICogMC41CiAgICBleWVfcmlnaHQgPSAoZV9yaSArIGVfcm8pICogMC41CgogICAgYmxvY2tbMDozXSA9IG5vc2UKICAgIGJsb2NrWzM6Nl0gPSBtX2xlZnQKICAgIGJsb2NrWzY6OV0gPSBtX3JpZ2h0CiAgICBibG9ja1s5OjEyXSA9IG1vdXRoX2NlbnRlcgogICAgYmxvY2tbMTI6MTVdID0gZV9sbwogICAgYmxvY2tbMTU6MThdID0gZV9saQogICAgYmxvY2tbMTg6MjFdID0gZV9yaQogICAgYmxvY2tbMjE6MjRdID0gZV9ybwogICAgYmxvY2tbMjRdID0gZmxvYXQobnAubGluYWxnLm5vcm0oZXllX2xlZnQgLSBleWVfcmlnaHQpKQogICAgYmxvY2tbMjVdID0gMS4wCiAgICByZXR1cm4gYmxvY2sKCgpkZWYgX2J1aWxkX2hhbmRmYWNlKGZhY2VfeHl6LCBhbmNob3IsIHNjYWxlLCBsZWZ0X3h5eiwgcmlnaHRfeHl6LCBmYWNlX3ByZXNlbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJhbmNob3IgcmVsICg5KSArIHBlci1oYW5kIHdyaXN0L3BhbG0tPmZhY2UgdmVjdG9ycyAoMjQpICsgNCBkaXN0ICsgMyBmbGFncy4iIiIKICAgIGJsb2NrID0gbnAuemVyb3MoNDAsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBoYXNfZmFjZSA9IGJvb2woZmFjZV9wcmVzZW50KQogICAgbGVmdF9wcmVzZW50ID0gbGVmdF94eXogaXMgbm90IE5vbmUKICAgIHJpZ2h0X3ByZXNlbnQgPSByaWdodF94eXogaXMgbm90IE5vbmUKCiAgICBpZiBoYXNfZmFjZToKICAgICAgICBub3NlX3JhdyA9IG5wLmFzYXJyYXkoZmFjZV94eXpbTk9TRV9USVBdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIG1vdXRoX3JhdyA9ICgKICAgICAgICAgICAgbnAuYXNhcnJheShmYWNlX3h5eltNT1VUSF9UT1BdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICAgICArIG5wLmFzYXJyYXkoZmFjZV94eXpbTU9VVEhfQk9UVE9NXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgKyBucC5hc2FycmF5KGZhY2VfeHl6W01PVVRIX0xFRlRdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICAgICArIG5wLmFzYXJyYXkoZmFjZV94eXpbTU9VVEhfUklHSFRdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICkgLyA0LjAKICAgICAgICBleWVfcmF3ID0gKAogICAgICAgICAgICBucC5hc2FycmF5KGZhY2VfeHl6W0VZRV9MX09VVEVSXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgKyBucC5hc2FycmF5KGZhY2VfeHl6W0VZRV9MX0lOTkVSXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgKyBucC5hc2FycmF5KGZhY2VfeHl6W0VZRV9SX0lOTkVSXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgKyBucC5hc2FycmF5KGZhY2VfeHl6W0VZRV9SX09VVEVSXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICApIC8gNC4wCiAgICAgICAgYmxvY2tbMDozXSA9IF9yZWwobm9zZV9yYXcsIGFuY2hvciwgc2NhbGUpCiAgICAgICAgYmxvY2tbMzo2XSA9IF9yZWwobW91dGhfcmF3LCBhbmNob3IsIHNjYWxlKQogICAgICAgIGJsb2NrWzY6OV0gPSBfcmVsKGV5ZV9yYXcsIGFuY2hvciwgc2NhbGUpCgogICAgICAgIGZvciBoYW5kX2lkeCwgaGFuZCBpbiBlbnVtZXJhdGUoKGxlZnRfeHl6LCByaWdodF94eXopKToKICAgICAgICAgICAgaWYgaGFuZCBpcyBOb25lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgYmFzZSA9IDkgKyBoYW5kX2lkeCAqIDEyCiAgICAgICAgICAgIHdyaXN0ID0gbnAuYXNhcnJheShoYW5kWzBdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICAgICBwYWxtID0gcGFsbV9jZW50ZXIobnAuYXNhcnJheShoYW5kLCBkdHlwZT1ucC5mbG9hdDMyKSkKICAgICAgICAgICAgYmxvY2tbYmFzZSArIDA6YmFzZSArIDNdID0gKG5vc2VfcmF3IC0gd3Jpc3QpIC8gc2NhbGUKICAgICAgICAgICAgYmxvY2tbYmFzZSArIDM6YmFzZSArIDZdID0gKG1vdXRoX3JhdyAtIHdyaXN0KSAvIHNjYWxlCiAgICAgICAgICAgIGJsb2NrW2Jhc2UgKyA2OmJhc2UgKyA5XSA9IChleWVfcmF3IC0gd3Jpc3QpIC8gc2NhbGUKICAgICAgICAgICAgYmxvY2tbYmFzZSArIDk6YmFzZSArIDEyXSA9IChub3NlX3JhdyAtIHBhbG0pIC8gc2NhbGUKCiAgICAgICAgaWYgbGVmdF9wcmVzZW50OgogICAgICAgICAgICBibG9ja1szM10gPSBmbG9hdChucC5saW5hbGcubm9ybSgobm9zZV9yYXcgLSBucC5hc2FycmF5KGxlZnRfeHl6WzBdLCBkdHlwZT1ucC5mbG9hdDMyKSkgLyBzY2FsZSkpCiAgICAgICAgICAgIGJsb2NrWzM1XSA9IGZsb2F0KG5wLmxpbmFsZy5ub3JtKChtb3V0aF9yYXcgLSBwYWxtX2NlbnRlcihucC5hc2FycmF5KGxlZnRfeHl6LCBkdHlwZT1ucC5mbG9hdDMyKSkpIC8gc2NhbGUpKQogICAgICAgIGlmIHJpZ2h0X3ByZXNlbnQ6CiAgICAgICAgICAgIGJsb2NrWzM0XSA9IGZsb2F0KG5wLmxpbmFsZy5ub3JtKChub3NlX3JhdyAtIG5wLmFzYXJyYXkocmlnaHRfeHl6WzBdLCBkdHlwZT1ucC5mbG9hdDMyKSkgLyBzY2FsZSkpCiAgICAgICAgICAgIGJsb2NrWzM2XSA9IGZsb2F0KG5wLmxpbmFsZy5ub3JtKChtb3V0aF9yYXcgLSBwYWxtX2NlbnRlcihucC5hc2FycmF5KHJpZ2h0X3h5eiwgZHR5cGU9bnAuZmxvYXQzMikpKSAvIHNjYWxlKSkKCiAgICBibG9ja1szN10gPSBmbG9hdChoYXNfZmFjZSkKICAgIGJsb2NrWzM4XSA9IGZsb2F0KGxlZnRfcHJlc2VudCkKICAgIGJsb2NrWzM5XSA9IGZsb2F0KHJpZ2h0X3ByZXNlbnQpCiAgICByZXR1cm4gYmxvY2sKCgpkZWYgX2J1aWxkX2hhbmRmYWNlX3ZlbChzbWFydF92ZWMxODAsIGhhbmRmYWNlX2Jsb2NrLCBzdGF0ZSwgZHQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJoYW5kZmFjZSg0MCkgKyBwZXItc2Vjb25kIHZlbG9jaXR5IG9mIHNtYXJ0MTgwIGxlZnQvcmlnaHQgZ2xvYmFsIHNsaWNlcyAoMzMrMzMpLiIiIgogICAgYmxvY2sgPSBucC56ZXJvcygxMDYsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBibG9ja1swOjQwXSA9IGhhbmRmYWNlX2Jsb2NrCiAgICBjdXJfbGVmdCA9IG5wLmFzYXJyYXkoc21hcnRfdmVjMTgwW19TTElDRV9MRUZUX0dMT0JBTF0sIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBjdXJfcmlnaHQgPSBucC5hc2FycmF5KHNtYXJ0X3ZlYzE4MFtfU0xJQ0VfUklHSFRfR0xPQkFMXSwgZHR5cGU9bnAuZmxvYXQzMikKICAgIGR0X3MgPSBtYXgoZmxvYXQoZHQpLCAxZS0zKQogICAgcHJldl9sZWZ0ID0gc3RhdGUuZ2V0KCJoYW5kZmFjZXZlbF9wcmV2X2xlZnQiKSBpZiBzdGF0ZSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgIHByZXZfcmlnaHQgPSBzdGF0ZS5nZXQoImhhbmRmYWNldmVsX3ByZXZfcmlnaHQiKSBpZiBzdGF0ZSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgIGlmIHByZXZfbGVmdCBpcyBub3QgTm9uZToKICAgICAgICBibG9ja1s0MDo3M10gPSAoY3VyX2xlZnQgLSBucC5hc2FycmF5KHByZXZfbGVmdCwgZHR5cGU9bnAuZmxvYXQzMikpIC8gZHRfcwogICAgaWYgcHJldl9yaWdodCBpcyBub3QgTm9uZToKICAgICAgICBibG9ja1s3MzoxMDZdID0gKGN1cl9yaWdodCAtIG5wLmFzYXJyYXkocHJldl9yaWdodCwgZHR5cGU9bnAuZmxvYXQzMikpIC8gZHRfcwogICAgaWYgc3RhdGUgaXMgbm90IE5vbmU6CiAgICAgICAgc3RhdGVbImhhbmRmYWNldmVsX3ByZXZfbGVmdCJdID0gY3VyX2xlZnQuY29weSgpCiAgICAgICAgc3RhdGVbImhhbmRmYWNldmVsX3ByZXZfcmlnaHQiXSA9IGN1cl9yaWdodC5jb3B5KCkKICAgIHJldHVybiBibG9jawoKCmRlZiBfc2NoZW1hX25hbWUoc2NoZW1hKSAtPiBzdHI6CiAgICBpZiBpc2luc3RhbmNlKHNjaGVtYSwgc3RyKToKICAgICAgICByZXR1cm4gc2NoZW1hCiAgICByZXR1cm4gZ2V0YXR0cihzY2hlbWEsICJuYW1lIiwgc3RyKHNjaGVtYSkpCgoKZGVmIGlzX2ZhY2VfcmVmX3NjaGVtYShzY2hlbWEpIC0+IGJvb2w6CiAgICByZXR1cm4gX3NjaGVtYV9uYW1lKHNjaGVtYSkgaW4gRkFDRV9SRUZfQkxPQ0tfRElNUwoKCmRlZiBidWlsZF9mYWNlX3NjaGVtYV92ZWN0b3IoCiAgICBzY2hlbWFfbmFtZSwKICAgIHNtYXJ0X3ZlYzE4MDogbnAubmRhcnJheSwKICAgIGZhY2VfeHl6OiBucC5uZGFycmF5IHwgTm9uZSwKICAgIHNob3VsZGVyczogbnAubmRhcnJheSwKICAgIGxlZnRfeHl6OiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsCiAgICByaWdodF94eXo6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSwKICAgIGZhY2VfcHJlc2VudDogYm9vbCA9IFRydWUsCiAgICBzdGF0ZTogZGljdCB8IE5vbmUgPSBOb25lLAogICAgZHQ6IGZsb2F0ID0gMC4xLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gYGBzbWFydDE4MFswOjE4MF1gYCBjb25jYXRlbmF0ZWQgd2l0aCB0aGUgZmFjZS1yZWYgYmxvY2sgZm9yIGBgc2NoZW1hX25hbWVgYC4KCiAgICBTaGFyZWQgYnkgdGhlIG9mZmxpbmUgTlBaIGNvbnZlcnRlciBhbmQgdGhlIGxpdmUgZXh0cmFjdG9yLiBgYHN0YXRlYGAgKGEgbXV0YWJsZQogICAgZGljdCB0aGUgY2FsbGVyIGtlZXBzIHBlciB2aWRlbyAvIHBlciBsaXZlIHNlc3Npb24pIGNhcnJpZXMgdmVsb2NpdHkgaGlzdG9yeTsKICAgIHBhc3MgYGBOb25lYGAgZm9yIHN0YXRlbGVzcyBvbmUtc2hvdCB1c2UgKHZlbG9jaXR5IHRlcm1zIGJlY29tZSAwKS4KICAgICIiIgogICAgbmFtZSA9IF9zY2hlbWFfbmFtZShzY2hlbWFfbmFtZSkKICAgIGlmIG5hbWUgbm90IGluIEZBQ0VfUkVGX0JMT0NLX0RJTVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkJ1a2FuIHNjaGVtYSBmYWNlLXJlZmVyZW5jZToge25hbWV9IikKCiAgICBzbWFydCA9IG5wLmFzYXJyYXkoc21hcnRfdmVjMTgwLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgaWYgc21hcnQuc2hhcGVbMF0gIT0gU01BUlRfQkFTRV9ESU06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInNtYXJ0X3ZlYzE4MCBoYXJ1cyB7U01BUlRfQkFTRV9ESU19LUQsIGdvdCB7c21hcnQuc2hhcGVbMF19IikKCiAgICBhbmNob3IsIHNjYWxlID0gX3Nob3VsZGVyX3JlZihzaG91bGRlcnMpCiAgICBmYWNlX29rID0gYm9vbCgKICAgICAgICBmYWNlX3ByZXNlbnQKICAgICAgICBhbmQgZmFjZV94eXogaXMgbm90IE5vbmUKICAgICAgICBhbmQgbnAuaXNmaW5pdGUobnAuYXNhcnJheShmYWNlX3h5eiwgZHR5cGU9bnAuZmxvYXQzMikpLmFsbCgpCiAgICApCgogICAgaWYgbmFtZSA9PSAic21hcnQxODBfbW91dGhkeW4yMTQiOgogICAgICAgIGJsb2NrID0gX2J1aWxkX21vdXRoZHluKGZhY2VfeHl6LCBhbmNob3IsIHNjYWxlLCBmYWNlX29rLCBzdGF0ZSwgZHQpCiAgICBlbGlmIG5hbWUgPT0gInNtYXJ0MTgwX21vdXRoc3RhdDIwNiI6CiAgICAgICAgYmxvY2sgPSBfYnVpbGRfbW91dGhzdGF0KGZhY2VfeHl6LCBhbmNob3IsIHNjYWxlLCBmYWNlX29rKQogICAgZWxpZiBuYW1lID09ICJzbWFydDE4MF9oYW5kZmFjZTIyMCI6CiAgICAgICAgYmxvY2sgPSBfYnVpbGRfaGFuZGZhY2UoZmFjZV94eXosIGFuY2hvciwgc2NhbGUsIGxlZnRfeHl6LCByaWdodF94eXosIGZhY2Vfb2spCiAgICBlbGlmIG5hbWUgPT0gInNtYXJ0MTgwX2hhbmRmYWNlX3ZlbDI4NiI6CiAgICAgICAgaGFuZGZhY2UgPSBfYnVpbGRfaGFuZGZhY2UoZmFjZV94eXosIGFuY2hvciwgc2NhbGUsIGxlZnRfeHl6LCByaWdodF94eXosIGZhY2Vfb2spCiAgICAgICAgYmxvY2sgPSBfYnVpbGRfaGFuZGZhY2VfdmVsKHNtYXJ0LCBoYW5kZmFjZSwgc3RhdGUsIGR0KQogICAgZWxzZTogICMgcHJhZ21hOiBubyBjb3ZlciAtIGd1YXJkZWQgYWJvdmUKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQnVrYW4gc2NoZW1hIGZhY2UtcmVmZXJlbmNlOiB7bmFtZX0iKQoKICAgIHZlY3RvciA9IG5wLmNvbmNhdGVuYXRlKFtzbWFydCwgYmxvY2tdKS5hc3R5cGUobnAuZmxvYXQzMikKICAgIHJldHVybiBucC5uYW5fdG9fbnVtKHZlY3RvciwgbmFuPTAuMCwgcG9zaW5mPTAuMCwgbmVnaW5mPTAuMCkKCgpkZWYgZmFjZV9ibG9ja19jb2x1bW5fbmFtZXMoc2NoZW1hX25hbWUpIC0+IGxpc3Rbc3RyXToKICAgICIiIkNvbHVtbiBuYW1lcyBmb3IgdGhlIGV4dHJhIGJsb2NrIG9ubHkgKHNtYXJ0MTgwIG5hbWVzIGhhbmRsZWQgYnkgZmVhdHVyZV9zY2hlbWFzKS4iIiIKICAgIG5hbWUgPSBfc2NoZW1hX25hbWUoc2NoZW1hX25hbWUpCiAgICBheGVzID0gKCJ4IiwgInkiLCAieiIpCiAgICBpZiBuYW1lID09ICJzbWFydDE4MF9tb3V0aGR5bjIxNCI6CiAgICAgICAgbmFtZXMgPSBbZiJmYWNlX3twdH1fe2F4fSIgZm9yIHB0IGluIE1PVVRIRFlOX1BPSU5UX05BTUVTIGZvciBheCBpbiBheGVzXQogICAgICAgIG5hbWVzICs9IFsibW91dGhfb3Blbm5lc3MiLCAibW91dGhfd2lkdGgiLCAibW91dGhfYXNwZWN0IiwgImV5ZV9kaXN0Il0KICAgICAgICBuYW1lcyArPSBbImRfbW91dGhfb3Blbm5lc3MiLCAiZF9tb3V0aF95IiwgImZhY2VfcHJlc2VudCJdCiAgICAgICAgcmV0dXJuIG5hbWVzCiAgICBpZiBuYW1lID09ICJzbWFydDE4MF9tb3V0aHN0YXQyMDYiOgogICAgICAgIG5hbWVzOiBsaXN0W3N0cl0gPSBbXQogICAgICAgIG5hbWVzICs9IFtmImZhY2Vfbm9zZV97YXh9IiBmb3IgYXggaW4gYXhlc10KICAgICAgICBuYW1lcyArPSBbZiJmYWNlX21vdXRoX2xlZnRfe2F4fSIgZm9yIGF4IGluIGF4ZXNdCiAgICAgICAgbmFtZXMgKz0gW2YiZmFjZV9tb3V0aF9yaWdodF97YXh9IiBmb3IgYXggaW4gYXhlc10KICAgICAgICBuYW1lcyArPSBbZiJmYWNlX21vdXRoX2NlbnRlcl97YXh9IiBmb3IgYXggaW4gYXhlc10KICAgICAgICBmb3IgY29ybmVyIGluICgiZXllX2xfb3V0ZXIiLCAiZXllX2xfaW5uZXIiLCAiZXllX3JfaW5uZXIiLCAiZXllX3Jfb3V0ZXIiKToKICAgICAgICAgICAgbmFtZXMgKz0gW2YiZmFjZV97Y29ybmVyfV97YXh9IiBmb3IgYXggaW4gYXhlc10KICAgICAgICBuYW1lcyArPSBbImV5ZV9kaXN0IiwgImZhY2VfcHJlc2VudCJdCiAgICAgICAgcmV0dXJuIG5hbWVzCiAgICBpZiBuYW1lIGluIHsic21hcnQxODBfaGFuZGZhY2UyMjAiLCAic21hcnQxODBfaGFuZGZhY2VfdmVsMjg2In06CiAgICAgICAgbmFtZXMgPSBbXQogICAgICAgIGZvciBhbmNob3JfbmFtZSBpbiAoIm5vc2UiLCAibW91dGhfY2VudGVyIiwgImV5ZV9jZW50ZXIiKToKICAgICAgICAgICAgbmFtZXMgKz0gW2YiZmFjZV9hbmNob3Jfe2FuY2hvcl9uYW1lfV97YXh9IiBmb3IgYXggaW4gYXhlc10KICAgICAgICBmb3IgaGFuZCBpbiAoImxlZnQiLCAicmlnaHQiKToKICAgICAgICAgICAgZm9yIHRhcmdldCBpbiAoIndyaXN0X3RvX25vc2UiLCAid3Jpc3RfdG9fbW91dGgiLCAid3Jpc3RfdG9fZXllIiwgInBhbG1fdG9fbm9zZSIpOgogICAgICAgICAgICAgICAgbmFtZXMgKz0gW2Yie2hhbmR9X3t0YXJnZXR9X3theH0iIGZvciBheCBpbiBheGVzXQogICAgICAgIG5hbWVzICs9IFsiZGlzdF9sZWZ0X3dyaXN0X25vc2UiLCAiZGlzdF9yaWdodF93cmlzdF9ub3NlIiwgImRpc3RfbGVmdF9wYWxtX21vdXRoIiwgImRpc3RfcmlnaHRfcGFsbV9tb3V0aCJdCiAgICAgICAgbmFtZXMgKz0gWyJmYWNlX3ByZXNlbnQiLCAibGVmdF9wcmVzZW50IiwgInJpZ2h0X3ByZXNlbnQiXQogICAgICAgIGlmIG5hbWUgPT0gInNtYXJ0MTgwX2hhbmRmYWNlX3ZlbDI4NiI6CiAgICAgICAgICAgIGZvciBoYW5kIGluICgibGVmdCIsICJyaWdodCIpOgogICAgICAgICAgICAgICAgZm9yIHBvaW50IGluIF9TTUFSVDE4MF9TRUxFQ1RFRF9QT0lOVFM6CiAgICAgICAgICAgICAgICAgICAgbmFtZXMgKz0gW2YidmVsX3toYW5kfV97cG9pbnR9X3theH0iIGZvciBheCBpbiBheGVzXQogICAgICAgIHJldHVybiBuYW1lcwogICAgcmFpc2UgVmFsdWVFcnJvcihmIkJ1a2FuIHNjaGVtYSBmYWNlLXJlZmVyZW5jZToge25hbWV9IikK', 'gru_adi.py': 'IiIiQWRpLXN0eWxlIEdSVSBhcmNoaXRlY3R1cmUgd2l0aCBSZUxVIGNhbmRpZGF0ZSBhY3RpdmF0aW9uLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKClRBUkdFVF9GUkFNRVMgPSA2MApERUZBVUxUX0xSID0gMWUtMwpERUZBVUxUX0JBVENIX1NJWkUgPSAzMgpERUZBVUxUX0VQT0NIUyA9IDMwMApERUZBVUxUX0RST1BPVVQgPSAwLjMwCkRFRkFVTFRfUEFUSUVOQ0UgPSA0MAoKCmNsYXNzIFJlTFVHUlVDZWxsKG5uLk1vZHVsZSk6CiAgICAiIiJHUlUgY2VsbCBtYXRjaGluZyBLZXJhcyBHUlUncyBjb25maWd1cmFibGUgY2FuZGlkYXRlIGFjdGl2YXRpb24uIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlucHV0X2RpbTogaW50LCBoaWRkZW5fZGltOiBpbnQsIGFjdGl2YXRpb246IHN0ciA9ICJyZWx1IikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmlucHV0X2RpbSA9IGludChpbnB1dF9kaW0pCiAgICAgICAgc2VsZi5oaWRkZW5fZGltID0gaW50KGhpZGRlbl9kaW0pCiAgICAgICAgc2VsZi5hY3RpdmF0aW9uX25hbWUgPSBhY3RpdmF0aW9uCiAgICAgICAgc2VsZi53ZWlnaHRfaWggPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkoMyAqIGhpZGRlbl9kaW0sIGlucHV0X2RpbSkpCiAgICAgICAgc2VsZi53ZWlnaHRfaGggPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkoMyAqIGhpZGRlbl9kaW0sIGhpZGRlbl9kaW0pKQogICAgICAgIHNlbGYuYmlhc19paCA9IG5uLlBhcmFtZXRlcih0b3JjaC5lbXB0eSgzICogaGlkZGVuX2RpbSkpCiAgICAgICAgc2VsZi5iaWFzX2hoID0gbm4uUGFyYW1ldGVyKHRvcmNoLmVtcHR5KDMgKiBoaWRkZW5fZGltKSkKICAgICAgICBzZWxmLnJlc2V0X3BhcmFtZXRlcnMoKQoKICAgIGRlZiByZXNldF9wYXJhbWV0ZXJzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi53ZWlnaHRfaWgpCiAgICAgICAgZm9yIGNodW5rIGluIHNlbGYud2VpZ2h0X2hoLmNodW5rKDMsIGRpbT0wKToKICAgICAgICAgICAgbm4uaW5pdC5vcnRob2dvbmFsXyhjaHVuaykKICAgICAgICBmYW5faW4gPSBzZWxmLmlucHV0X2RpbSArIHNlbGYuaGlkZGVuX2RpbQogICAgICAgIGJvdW5kID0gMS4wIC8gbWF0aC5zcXJ0KGZhbl9pbikKICAgICAgICBubi5pbml0LnVuaWZvcm1fKHNlbGYuYmlhc19paCwgLWJvdW5kLCBib3VuZCkKICAgICAgICBubi5pbml0LnVuaWZvcm1fKHNlbGYuYmlhc19oaCwgLWJvdW5kLCBib3VuZCkKCiAgICBkZWYgX2NhbmRpZGF0ZV9hY3RpdmF0aW9uKHNlbGYsIHZhbHVlOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBzZWxmLmFjdGl2YXRpb25fbmFtZSA9PSAicmVsdSI6CiAgICAgICAgICAgIHJldHVybiB0b3JjaC5yZWx1KHZhbHVlKQogICAgICAgIGlmIHNlbGYuYWN0aXZhdGlvbl9uYW1lID09ICJ0YW5oIjoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnRhbmgodmFsdWUpCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuc3VwcG9ydGVkIEdSVSBhY3RpdmF0aW9uOiB7c2VsZi5hY3RpdmF0aW9uX25hbWV9IikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IsIGhfcHJldjogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgZ2kgPSB0b3JjaC5tYXRtdWwoeCwgc2VsZi53ZWlnaHRfaWgudCgpKSArIHNlbGYuYmlhc19paAogICAgICAgIGdoID0gdG9yY2gubWF0bXVsKGhfcHJldiwgc2VsZi53ZWlnaHRfaGgudCgpKSArIHNlbGYuYmlhc19oaAogICAgICAgIGlfeiwgaV9yLCBpX24gPSBnaS5jaHVuaygzLCBkaW09LTEpCiAgICAgICAgaF96LCBoX3IsIGhfbiA9IGdoLmNodW5rKDMsIGRpbT0tMSkKCiAgICAgICAgeiA9IHRvcmNoLnNpZ21vaWQoaV96ICsgaF96KQogICAgICAgIHIgPSB0b3JjaC5zaWdtb2lkKGlfciArIGhfcikKICAgICAgICBuID0gc2VsZi5fY2FuZGlkYXRlX2FjdGl2YXRpb24oaV9uICsgciAqIGhfbikKICAgICAgICByZXR1cm4geiAqIGhfcHJldiArICgxLjAgLSB6KSAqIG4KCgpjbGFzcyBSZUxVR1JVTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIlNpbmdsZS1kaXJlY3Rpb24gYmF0Y2gtZmlyc3QgR1JVIGxheWVyIHdpdGggUmVMVSBjYW5kaWRhdGUgYWN0aXZhdGlvbi4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRfZGltOiBpbnQsIGhpZGRlbl9kaW06IGludCwgcmV0dXJuX3NlcXVlbmNlczogYm9vbCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmhpZGRlbl9kaW0gPSBpbnQoaGlkZGVuX2RpbSkKICAgICAgICBzZWxmLnJldHVybl9zZXF1ZW5jZXMgPSBib29sKHJldHVybl9zZXF1ZW5jZXMpCiAgICAgICAgc2VsZi5jZWxsID0gUmVMVUdSVUNlbGwoaW5wdXRfZGltPWlucHV0X2RpbSwgaGlkZGVuX2RpbT1oaWRkZW5fZGltLCBhY3RpdmF0aW9uPSJyZWx1IikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBiYXRjaF9zaXplLCBzdGVwcywgXyA9IHguc2hhcGUKICAgICAgICBoID0geC5uZXdfemVyb3MoYmF0Y2hfc2l6ZSwgc2VsZi5oaWRkZW5fZGltKQogICAgICAgIG91dHB1dHMgPSBbXQogICAgICAgIGZvciBzdGVwIGluIHJhbmdlKHN0ZXBzKToKICAgICAgICAgICAgaCA9IHNlbGYuY2VsbCh4WzosIHN0ZXAsIDpdLCBoKQogICAgICAgICAgICBpZiBzZWxmLnJldHVybl9zZXF1ZW5jZXM6CiAgICAgICAgICAgICAgICBvdXRwdXRzLmFwcGVuZChoKQogICAgICAgIGlmIHNlbGYucmV0dXJuX3NlcXVlbmNlczoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnN0YWNrKG91dHB1dHMsIGRpbT0xKQogICAgICAgIHJldHVybiBoCgoKZGVmIF9iYXRjaF9ub3JtX3RpbWUoYmF0Y2hfbm9ybTogbm4uQmF0Y2hOb3JtMWQsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgcmV0dXJuIGJhdGNoX25vcm0oeC50cmFuc3Bvc2UoMSwgMikpLnRyYW5zcG9zZSgxLCAyKQoKCmNsYXNzIEFkaUdSVU1vZGVsKG5uLk1vZHVsZSk6CiAgICAiIiJUd28tbGF5ZXIgUmVMVS1HUlUgbmV0d29yayBhZGFwdGVkIGZyb20gQWRpIGV0IGFsLidzIHBhcGVyLiIiIgoKICAgIHRhcmdldF9mcmFtZXMgPSBUQVJHRVRfRlJBTUVTCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfZGltOiBpbnQgPSAxODAsCiAgICAgICAgbnVtX2NsYXNzZXM6IGludCA9IDEwLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VULAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuaW5wdXRfZGltID0gaW50KGlucHV0X2RpbSkKICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gaW50KG51bV9jbGFzc2VzKQoKICAgICAgICBzZWxmLmdydTEgPSBSZUxVR1JVTGF5ZXIoc2VsZi5pbnB1dF9kaW0sIDY0LCByZXR1cm5fc2VxdWVuY2VzPVRydWUpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0xZCg2NCkKICAgICAgICBzZWxmLmRyb3BvdXQxID0gbm4uRHJvcG91dChmbG9hdChkcm9wb3V0KSkKCiAgICAgICAgc2VsZi5ncnUyID0gUmVMVUdSVUxheWVyKDY0LCAxMjgsIHJldHVybl9zZXF1ZW5jZXM9RmFsc2UpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0xZCgxMjgpCiAgICAgICAgc2VsZi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dCkpCgogICAgICAgIHNlbGYuZGVuc2UgPSBubi5MaW5lYXIoMTI4LCA2NCkKICAgICAgICBzZWxmLmRyb3BvdXQzID0gbm4uRHJvcG91dChmbG9hdChkcm9wb3V0KSkKICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5MaW5lYXIoNjQsIHNlbGYubnVtX2NsYXNzZXMpCiAgICAgICAgc2VsZi5hY3RpdmF0aW9uID0gbm4uUmVMVSgpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgeCA9IHNlbGYuZ3J1MSh4KQogICAgICAgIHggPSBfYmF0Y2hfbm9ybV90aW1lKHNlbGYuYm4xLCB4KQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQxKHgpCgogICAgICAgIHggPSBzZWxmLmdydTIoeCkKICAgICAgICB4ID0gc2VsZi5ibjIoeCkKICAgICAgICB4ID0gc2VsZi5kcm9wb3V0Mih4KQoKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UoeCkpCiAgICAgICAgeCA9IHNlbGYuZHJvcG91dDMoeCkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHgpCgoKZGVmIGJ1aWxkX21vZGVsKGlucHV0X2RpbTogaW50LCBudW1fY2xhc3NlczogaW50KSAtPiBBZGlHUlVNb2RlbDoKICAgIHJldHVybiBBZGlHUlVNb2RlbChpbnB1dF9kaW09aW5wdXRfZGltLCBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcykK', 'gru_khukuh.py': 'IiIiS2h1a3VoLXN0eWxlIEdSVSBhcmNoaXRlY3R1cmUgZm9yIFNtYXJ0IFY4IEJJU0lORE8gZmVhdHVyZXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpUQVJHRVRfRlJBTUVTID0gMzAKREVGQVVMVF9MUiA9IDFlLTQKREVGQVVMVF9CQVRDSF9TSVpFID0gNjQKREVGQVVMVF9FUE9DSFMgPSAxMDAKREVGQVVMVF9EUk9QT1VUX0lOUFVUID0gMC4xMApERUZBVUxUX0RST1BPVVRfQkxPQ0sgPSAwLjIwCgoKY2xhc3MgS2h1a3VoR1JVTW9kZWwobm4uTW9kdWxlKToKICAgICIiIlRocmVlLXN0YWdlIEdSVSBuZXR3b3JrIGFkYXB0ZWQgZnJvbSBLaHVrdWggUHJpaGF0bWlraG8ncyBwYXBlci4KCiAgICBUaGUgcGFwZXIgdXNlcyAzMCBmcmFtZXMgYW5kIDE2MjkgTWVkaWFQaXBlIEhvbGlzdGljIHZhbHVlcyBwZXIgZnJhbWUuIFRoaXMKICAgIHZlcnNpb24gcHJlc2VydmVzIHRoZSB0ZW1wb3JhbCBsYXlvdXQgYW5kIGxheWVyIHNpemVzLCB3aGlsZSBhY2NlcHRpbmcgdGhlCiAgICBsb2NhbCBTbWFydCBWOCAxODAtRCBmZWF0dXJlIGNvbnRyYWN0IHVzZWQgYnkgdGhlIGN1cnJlbnQgZGF0YXNldC4KICAgICIiIgoKICAgIHRhcmdldF9mcmFtZXMgPSBUQVJHRVRfRlJBTUVTCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfZGltOiBpbnQgPSAxODAsCiAgICAgICAgbnVtX2NsYXNzZXM6IGludCA9IDEwLAogICAgICAgIGRyb3BvdXRfaW5wdXQ6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VUX0lOUFVULAogICAgICAgIGRyb3BvdXRfYmxvY2s6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VUX0JMT0NLLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuaW5wdXRfZGltID0gaW50KGlucHV0X2RpbSkKICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gaW50KG51bV9jbGFzc2VzKQoKICAgICAgICBzZWxmLmlucHV0X2Ryb3BvdXQgPSBubi5Ecm9wb3V0KGZsb2F0KGRyb3BvdXRfaW5wdXQpKQoKICAgICAgICBzZWxmLmdydTEgPSBubi5HUlUoc2VsZi5pbnB1dF9kaW0sIDEyOCwgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLmRlbnNlMSA9IG5uLkxpbmVhcigxMjgsIDY0KQogICAgICAgIHNlbGYuZHJvcG91dDEgPSBubi5Ecm9wb3V0KGZsb2F0KGRyb3BvdXRfYmxvY2spKQoKICAgICAgICBzZWxmLmdydTIgPSBubi5HUlUoNjQsIDY0LCBiYXRjaF9maXJzdD1UcnVlKQogICAgICAgIHNlbGYuZGVuc2UyID0gbm4uTGluZWFyKDY0LCA2NCkKICAgICAgICBzZWxmLmRyb3BvdXQyID0gbm4uRHJvcG91dChmbG9hdChkcm9wb3V0X2Jsb2NrKSkKCiAgICAgICAgc2VsZi5ncnUzID0gbm4uR1JVKDY0LCAzMiwgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLmRlbnNlMyA9IG5uLkxpbmVhcigzMiwgNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MyA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuYmF0Y2hfbm9ybSA9IG5uLkJhdGNoTm9ybTFkKDY0KQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcig2NCwgc2VsZi5udW1fY2xhc3NlcykKICAgICAgICBzZWxmLmFjdGl2YXRpb24gPSBubi5SZUxVKCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICB4ID0gc2VsZi5pbnB1dF9kcm9wb3V0KHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTEoeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UxKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQxKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTIoeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UyKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQyKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTMoeCkKICAgICAgICB4ID0geFs6LCAtMSwgOl0KCiAgICAgICAgeCA9IHNlbGYuYWN0aXZhdGlvbihzZWxmLmRlbnNlMyh4KSkKICAgICAgICB4ID0gc2VsZi5kcm9wb3V0Myh4KQogICAgICAgIHggPSBzZWxmLmJhdGNoX25vcm0oeCkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHgpCgoKZGVmIGJ1aWxkX21vZGVsKGlucHV0X2RpbTogaW50LCBudW1fY2xhc3NlczogaW50KSAtPiBLaHVrdWhHUlVNb2RlbDoKICAgIHJldHVybiBLaHVrdWhHUlVNb2RlbChpbnB1dF9kaW09aW5wdXRfZGltLCBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcykK', 'gru_hybrid.py': 'IiIiSHlicmlkIEdSVSBhcmNoaXRlY3R1cmUgY29tYmluaW5nIEtodWt1aCBkZXB0aCB3aXRoIEFkaSByZWd1bGFyaXphdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKClRBUkdFVF9GUkFNRVMgPSAzMApERUZBVUxUX0xSID0gMWUtNApERUZBVUxUX0JBVENIX1NJWkUgPSA2NApERUZBVUxUX0VQT0NIUyA9IDE1MApERUZBVUxUX0RST1BPVVRfSU5QVVQgPSAwLjEwCkRFRkFVTFRfRFJPUE9VVF9CTE9DSyA9IDAuMzAKREVGQVVMVF9QQVRJRU5DRSA9IDMwCgoKZGVmIF9iYXRjaF9ub3JtX3RpbWUoYmF0Y2hfbm9ybTogbm4uQmF0Y2hOb3JtMWQsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgcmV0dXJuIGJhdGNoX25vcm0oeC50cmFuc3Bvc2UoMSwgMikpLnRyYW5zcG9zZSgxLCAyKQoKCmNsYXNzIEh5YnJpZEdSVU1vZGVsKG5uLk1vZHVsZSk6CiAgICAiIiJGYXN0IDMwLWZyYW1lIGh5YnJpZCBmb3IgbGl2ZSB1c2UuCgogICAgVGhlIG1vZGVsIGtlZXBzIEtodWt1aCdzIDEyOC82NC8zMiByZWN1cnJlbnQgZGVwdGggYW5kIGluc2VydHMgQWRpLXN0eWxlCiAgICBCYXRjaE5vcm0vRHJvcG91dCBhZnRlciByZWN1cnJlbnQgYmxvY2tzIGZvciBiZXR0ZXIgc3RhYmlsaXR5IG9uIHRoZSBzbWFsbGVyCiAgICBsb2NhbCBkYXRhc2V0LgogICAgIiIiCgogICAgdGFyZ2V0X2ZyYW1lcyA9IFRBUkdFVF9GUkFNRVMKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9kaW06IGludCA9IDE4MCwKICAgICAgICBudW1fY2xhc3NlczogaW50ID0gMTAsCiAgICAgICAgZHJvcG91dF9pbnB1dDogZmxvYXQgPSBERUZBVUxUX0RST1BPVVRfSU5QVVQsCiAgICAgICAgZHJvcG91dF9ibG9jazogZmxvYXQgPSBERUZBVUxUX0RST1BPVVRfQkxPQ0ssCiAgICApIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5pbnB1dF9kaW0gPSBpbnQoaW5wdXRfZGltKQogICAgICAgIHNlbGYubnVtX2NsYXNzZXMgPSBpbnQobnVtX2NsYXNzZXMpCgogICAgICAgIHNlbGYuaW5wdXRfZHJvcG91dCA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9pbnB1dCkpCgogICAgICAgIHNlbGYuZ3J1MSA9IG5uLkdSVShzZWxmLmlucHV0X2RpbSwgMTI4LCBiYXRjaF9maXJzdD1UcnVlKQogICAgICAgIHNlbGYuYm4xID0gbm4uQmF0Y2hOb3JtMWQoMTI4KQogICAgICAgIHNlbGYuZGVuc2UxID0gbm4uTGluZWFyKDEyOCwgNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MSA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuZ3J1MiA9IG5uLkdSVSg2NCwgNjQsIGJhdGNoX2ZpcnN0PVRydWUpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0xZCg2NCkKICAgICAgICBzZWxmLmRlbnNlMiA9IG5uLkxpbmVhcig2NCwgNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuZ3J1MyA9IG5uLkdSVSg2NCwgMzIsIGJhdGNoX2ZpcnN0PVRydWUpCiAgICAgICAgc2VsZi5kZW5zZTMgPSBubi5MaW5lYXIoMzIsIDY0KQogICAgICAgIHNlbGYuYm4zID0gbm4uQmF0Y2hOb3JtMWQoNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MyA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuYWN0aXZhdGlvbiA9IG5uLlJlTFUoKQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcig2NCwgc2VsZi5udW1fY2xhc3NlcykKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICB4ID0gc2VsZi5pbnB1dF9kcm9wb3V0KHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTEoeCkKICAgICAgICB4ID0gX2JhdGNoX25vcm1fdGltZShzZWxmLmJuMSwgeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UxKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQxKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTIoeCkKICAgICAgICB4ID0gX2JhdGNoX25vcm1fdGltZShzZWxmLmJuMiwgeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UyKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQyKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTMoeCkKICAgICAgICB4ID0geFs6LCAtMSwgOl0KICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UzKHgpKQogICAgICAgIHggPSBzZWxmLmJuMyh4KQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQzKHgpCiAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3NpZmllcih4KQoKCmRlZiBidWlsZF9tb2RlbChpbnB1dF9kaW06IGludCwgbnVtX2NsYXNzZXM6IGludCkgLT4gSHlicmlkR1JVTW9kZWw6CiAgICByZXR1cm4gSHlicmlkR1JVTW9kZWwoaW5wdXRfZGltPWlucHV0X2RpbSwgbnVtX2NsYXNzZXM9bnVtX2NsYXNzZXMpCg==', 'gru_biattn.py': 'IiIiQmlkaXJlY3Rpb25hbCBHUlUgd2l0aCB0ZW1wb3JhbCBhdHRlbnRpb24gcG9vbGluZy4KCkNvbXBhY3Qgc2VxdWVuY2UgY2xhc3NpZmllciAofjAuMk0gcGFyYW1zIGF0IDE4MC1EIGlucHV0KTogTGF5ZXJOb3JtIGZyb250LAphIHNpbmdsZSBiaWRpcmVjdGlvbmFsIGN1RE5OIEdSVSwgdGhlbiBhdHRlbnRpb24gcG9vbGluZyBvdmVyIHRpbWUgaW5zdGVhZCBvZgp0YWtpbmcgb25seSB0aGUgbGFzdCBoaWRkZW4gc3RhdGUg4oCUIHNvIGZyYW1lcyB0aGF0IG1hdHRlciBtb3N0IGZvciB0aGUgc2lnbgpkb21pbmF0ZSB0aGUgcG9vbGVkIHJlcHJlc2VudGF0aW9uLgoKTW9kdWxlIGNvbnRyYWN0IChzaGFyZWQgd2l0aCBncnVfYWRpIGV0Yy4pOiBleHBvc2VzIFRBUkdFVF9GUkFNRVMsCkRFRkFVTFRfTFIvQkFUQ0hfU0laRS9FUE9DSFMvUEFUSUVOQ0UgYW5kIGBgYnVpbGRfbW9kZWwoaW5wdXRfZGltLCBudW1fY2xhc3NlcylgYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpUQVJHRVRfRlJBTUVTID0gNjAKREVGQVVMVF9MUiA9IDFlLTMKREVGQVVMVF9CQVRDSF9TSVpFID0gMzIKREVGQVVMVF9FUE9DSFMgPSAzMDAKREVGQVVMVF9EUk9QT1VUID0gMC4zMApERUZBVUxUX1BBVElFTkNFID0gNDAKCkhJRERFTl9ESU0gPSA5NgoKCmNsYXNzIEJpR1JVQXR0ZW50aW9uTW9kZWwobm4uTW9kdWxlKToKICAgICIiIkxheWVyTm9ybSAtPiBCaUdSVSAtPiBhZGRpdGl2ZSBhdHRlbnRpb24gcG9vbGluZyAtPiBGQyBjbGFzc2lmaWVyLiIiIgoKICAgIHRhcmdldF9mcmFtZXMgPSBUQVJHRVRfRlJBTUVTCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfZGltOiBpbnQgPSAxODAsCiAgICAgICAgbnVtX2NsYXNzZXM6IGludCA9IDEwLAogICAgICAgIGhpZGRlbl9kaW06IGludCA9IEhJRERFTl9ESU0sCiAgICAgICAgZHJvcG91dDogZmxvYXQgPSBERUZBVUxUX0RST1BPVVQsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5pbnB1dF9kaW0gPSBpbnQoaW5wdXRfZGltKQogICAgICAgIHNlbGYubnVtX2NsYXNzZXMgPSBpbnQobnVtX2NsYXNzZXMpCiAgICAgICAgc2VsZi5oaWRkZW5fZGltID0gaW50KGhpZGRlbl9kaW0pCgogICAgICAgIHNlbGYuaW5wdXRfbm9ybSA9IG5uLkxheWVyTm9ybShzZWxmLmlucHV0X2RpbSkKICAgICAgICBzZWxmLmdydSA9IG5uLkdSVSgKICAgICAgICAgICAgaW5wdXRfc2l6ZT1zZWxmLmlucHV0X2RpbSwKICAgICAgICAgICAgaGlkZGVuX3NpemU9c2VsZi5oaWRkZW5fZGltLAogICAgICAgICAgICBudW1fbGF5ZXJzPTEsCiAgICAgICAgICAgIGJhdGNoX2ZpcnN0PVRydWUsCiAgICAgICAgICAgIGJpZGlyZWN0aW9uYWw9VHJ1ZSwKICAgICAgICApCiAgICAgICAgZmVhdF9kaW0gPSBzZWxmLmhpZGRlbl9kaW0gKiAyCiAgICAgICAgc2VsZi5hdHRuID0gbm4uTGluZWFyKGZlYXRfZGltLCAxKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dCkpCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uTGluZWFyKGZlYXRfZGltLCBzZWxmLm51bV9jbGFzc2VzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHggPSBzZWxmLmlucHV0X25vcm0oeCkKICAgICAgICBzZXEsIF8gPSBzZWxmLmdydSh4KSAgICAgICAgICAgICAgICAgICAgICAgIyAoQiwgVCwgMkgpCiAgICAgICAgc2NvcmVzID0gc2VsZi5hdHRuKHNlcSkgICAgICAgICAgICAgICAgICAgICMgKEIsIFQsIDEpCiAgICAgICAgd2VpZ2h0cyA9IHRvcmNoLnNvZnRtYXgoc2NvcmVzLCBkaW09MSkgICAgICMgYXR0ZW50aW9uIG92ZXIgdGltZQogICAgICAgIHBvb2xlZCA9IHRvcmNoLnN1bSh3ZWlnaHRzICogc2VxLCBkaW09MSkgICAjIChCLCAySCkKICAgICAgICBwb29sZWQgPSBzZWxmLmRyb3BvdXQocG9vbGVkKQogICAgICAgIHJldHVybiBzZWxmLmNsYXNzaWZpZXIocG9vbGVkKQoKCmRlZiBidWlsZF9tb2RlbChpbnB1dF9kaW06IGludCwgbnVtX2NsYXNzZXM6IGludCkgLT4gQmlHUlVBdHRlbnRpb25Nb2RlbDoKICAgIHJldHVybiBCaUdSVUF0dGVudGlvbk1vZGVsKGlucHV0X2RpbT1pbnB1dF9kaW0sIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQo=', 'gru_convfront.py': 'IiIiQ29udjFkIGZyb250LWVuZCArIEdSVSBzZXF1ZW5jZSBjbGFzc2lmaWVyLgoKVHdvIHRlbXBvcmFsIENvbnYxZCBsYXllcnMgZXh0cmFjdCBsb2NhbCBtb3Rpb24gcGF0dGVybnMgYW5kIGhhbHZlIHRoZSB0aW1lIGF4aXMKKDYwIC0+IDMwKSBiZWZvcmUgYSBjdUROTiBHUlUgc3VtbWFyaXNlcyB0aGUgc2VxdWVuY2UgKH4wLjI3TSBwYXJhbXMgYXQgMTgwLUQpLgpUaGUgY29udiBmcm9udC1lbmQgZGVub2lzZXMgZnJhbWUtbGV2ZWwgaml0dGVyIGFuZCBsaWdodGVucyB0aGUgcmVjdXJyZW50IGxvYWQuCgpNb2R1bGUgY29udHJhY3QgKHNoYXJlZCB3aXRoIGdydV9hZGkgZXRjLik6IGV4cG9zZXMgVEFSR0VUX0ZSQU1FUywKREVGQVVMVF9MUi9CQVRDSF9TSVpFL0VQT0NIUy9QQVRJRU5DRSBhbmQgYGBidWlsZF9tb2RlbChpbnB1dF9kaW0sIG51bV9jbGFzc2VzKWBgLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKClRBUkdFVF9GUkFNRVMgPSA2MApERUZBVUxUX0xSID0gMWUtMwpERUZBVUxUX0JBVENIX1NJWkUgPSAzMgpERUZBVUxUX0VQT0NIUyA9IDMwMApERUZBVUxUX0RST1BPVVQgPSAwLjMwCkRFRkFVTFRfUEFUSUVOQ0UgPSA0MAoKQ09OVl9DSEFOTkVMUyA9IDEyOApHUlVfSElEREVOID0gMTI4CgoKY2xhc3MgQ29udkZyb250R1JVTW9kZWwobm4uTW9kdWxlKToKICAgICIiIkNvbnYxZChrNSkgLT4gQ29udjFkKGszKSAtPiBNYXhQb29sKDIpIC0+IEdSVSAtPiBGQyBjbGFzc2lmaWVyLiIiIgoKICAgIHRhcmdldF9mcmFtZXMgPSBUQVJHRVRfRlJBTUVTCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfZGltOiBpbnQgPSAxODAsCiAgICAgICAgbnVtX2NsYXNzZXM6IGludCA9IDEwLAogICAgICAgIGNvbnZfY2hhbm5lbHM6IGludCA9IENPTlZfQ0hBTk5FTFMsCiAgICAgICAgZ3J1X2hpZGRlbjogaW50ID0gR1JVX0hJRERFTiwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IERFRkFVTFRfRFJPUE9VVCwKICAgICkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmlucHV0X2RpbSA9IGludChpbnB1dF9kaW0pCiAgICAgICAgc2VsZi5udW1fY2xhc3NlcyA9IGludChudW1fY2xhc3NlcykKCiAgICAgICAgc2VsZi5jb252MSA9IG5uLkNvbnYxZChzZWxmLmlucHV0X2RpbSwgY29udl9jaGFubmVscywga2VybmVsX3NpemU9NSwgcGFkZGluZz0yKQogICAgICAgIHNlbGYuY29udjIgPSBubi5Db252MWQoY29udl9jaGFubmVscywgY29udl9jaGFubmVscywga2VybmVsX3NpemU9MywgcGFkZGluZz0xKQogICAgICAgIHNlbGYucG9vbCA9IG5uLk1heFBvb2wxZChrZXJuZWxfc2l6ZT0yKQogICAgICAgIHNlbGYuYWN0ID0gbm4uUmVMVSgpCiAgICAgICAgc2VsZi5jb252X2Ryb3BvdXQgPSBubi5Ecm9wb3V0KGZsb2F0KGRyb3BvdXQpKQoKICAgICAgICBzZWxmLmdydSA9IG5uLkdSVSgKICAgICAgICAgICAgaW5wdXRfc2l6ZT1jb252X2NoYW5uZWxzLAogICAgICAgICAgICBoaWRkZW5fc2l6ZT1ncnVfaGlkZGVuLAogICAgICAgICAgICBudW1fbGF5ZXJzPTEsCiAgICAgICAgICAgIGJhdGNoX2ZpcnN0PVRydWUsCiAgICAgICAgKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dCkpCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uTGluZWFyKGdydV9oaWRkZW4sIHNlbGYubnVtX2NsYXNzZXMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIyAoQiwgVCwgRCkgLT4gKEIsIEQsIFQpIGZvciBDb252MWQgb3ZlciB0aGUgdGltZSBheGlzLgogICAgICAgIHggPSB4LnRyYW5zcG9zZSgxLCAyKQogICAgICAgIHggPSBzZWxmLmFjdChzZWxmLmNvbnYxKHgpKQogICAgICAgIHggPSBzZWxmLmFjdChzZWxmLmNvbnYyKHgpKQogICAgICAgIHggPSBzZWxmLnBvb2woeCkgICAgICAgICAgICAgICAgICAjIChCLCBDLCBULzIpCiAgICAgICAgeCA9IHNlbGYuY29udl9kcm9wb3V0KHgpCiAgICAgICAgeCA9IHgudHJhbnNwb3NlKDEsIDIpICAgICAgICAgICAgICMgKEIsIFQvMiwgQykKICAgICAgICBfLCBoID0gc2VsZi5ncnUoeCkgICAgICAgICAgICAgICAgIyBoOiAoMSwgQiwgSCkKICAgICAgICBmZWF0ID0gc2VsZi5kcm9wb3V0KGhbLTFdKQogICAgICAgIHJldHVybiBzZWxmLmNsYXNzaWZpZXIoZmVhdCkKCgpkZWYgYnVpbGRfbW9kZWwoaW5wdXRfZGltOiBpbnQsIG51bV9jbGFzc2VzOiBpbnQpIC0+IENvbnZGcm9udEdSVU1vZGVsOgogICAgcmV0dXJuIENvbnZGcm9udEdSVU1vZGVsKGlucHV0X2RpbT1pbnB1dF9kaW0sIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQo=', 'tcn_sign.py': 'IiIiVGVtcG9yYWwgQ29udm9sdXRpb25hbCBOZXR3b3JrIChUQ04pIGZvciBCSVNJTkRPIHNpZ24gY2xhc3NpZmljYXRpb24uCgpGb3VyIHJlc2lkdWFsIGRpbGF0ZWQtY2F1c2FsIENvbnYxZCBibG9ja3MgKGRpbGF0aW9ucyAxLDIsNCw4LCBrZXJuZWwgMykgZ3JvdyB0aGUKcmVjZXB0aXZlIGZpZWxkIHRvIGNvdmVyIHRoZSB3aG9sZSA2MC1mcmFtZSB3aW5kb3cgd2l0aG91dCByZWN1cnJlbmNlLCB0aGVuIGEKZ2xvYmFsIGF2ZXJhZ2UgcG9vbCBmZWVkcyB0aGUgY2xhc3NpZmllciAofjAuNDNNIHBhcmFtcyBhdCAxODAtRCkuIEZ1bGx5CmNvbnZvbHV0aW9uYWwsIHNvIGl0IHBhcmFsbGVsaXNlcyB3ZWxsIGFuZCBpcyBmcmllbmRseSB0byB0aGUgSmV0c29uIEdQVS4KCk1vZHVsZSBjb250cmFjdCAoc2hhcmVkIHdpdGggZ3J1X2FkaSBldGMuKTogZXhwb3NlcyBUQVJHRVRfRlJBTUVTLApERUZBVUxUX0xSL0JBVENIX1NJWkUvRVBPQ0hTL1BBVElFTkNFIGFuZCBgYGJ1aWxkX21vZGVsKGlucHV0X2RpbSwgbnVtX2NsYXNzZXMpYGAuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCgoKVEFSR0VUX0ZSQU1FUyA9IDYwCkRFRkFVTFRfTFIgPSAxZS0zCkRFRkFVTFRfQkFUQ0hfU0laRSA9IDMyCkRFRkFVTFRfRVBPQ0hTID0gMzAwCkRFRkFVTFRfRFJPUE9VVCA9IDAuMjAKREVGQVVMVF9QQVRJRU5DRSA9IDQwCgpDSEFOTkVMUyA9IDEyOApLRVJORUxfU0laRSA9IDMKRElMQVRJT05TID0gKDEsIDIsIDQsIDgpCgoKY2xhc3MgX0Nob21wMWQobm4uTW9kdWxlKToKICAgICIiIlRyaW0gdGhlIHJpZ2h0IHBhZGRpbmcgc28gZWFjaCBibG9jayBzdGF5cyBjYXVzYWwuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNob21wX3NpemU6IGludCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNob21wX3NpemUgPSBpbnQoY2hvbXBfc2l6ZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBzZWxmLmNob21wX3NpemUgPT0gMDoKICAgICAgICAgICAgcmV0dXJuIHgKICAgICAgICByZXR1cm4geFs6LCA6LCA6IC1zZWxmLmNob21wX3NpemVdLmNvbnRpZ3VvdXMoKQoKCmNsYXNzIF9UZW1wb3JhbEJsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2g6IGludCwgb3V0X2NoOiBpbnQsIGtlcm5lbF9zaXplOiBpbnQsIGRpbGF0aW9uOiBpbnQsIGRyb3BvdXQ6IGZsb2F0KSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHBhZCA9IChrZXJuZWxfc2l6ZSAtIDEpICogZGlsYXRpb24KICAgICAgICBzZWxmLmNvbnYxID0gbm4uQ29udjFkKGluX2NoLCBvdXRfY2gsIGtlcm5lbF9zaXplLCBwYWRkaW5nPXBhZCwgZGlsYXRpb249ZGlsYXRpb24pCiAgICAgICAgc2VsZi5jaG9tcDEgPSBfQ2hvbXAxZChwYWQpCiAgICAgICAgc2VsZi5jb252MiA9IG5uLkNvbnYxZChvdXRfY2gsIG91dF9jaCwga2VybmVsX3NpemUsIHBhZGRpbmc9cGFkLCBkaWxhdGlvbj1kaWxhdGlvbikKICAgICAgICBzZWxmLmNob21wMiA9IF9DaG9tcDFkKHBhZCkKICAgICAgICBzZWxmLmFjdCA9IG5uLlJlTFUoKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dCkpCiAgICAgICAgc2VsZi5kb3duc2FtcGxlID0gbm4uQ29udjFkKGluX2NoLCBvdXRfY2gsIDEpIGlmIGluX2NoICE9IG91dF9jaCBlbHNlIE5vbmUKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBvdXQgPSBzZWxmLmRyb3BvdXQoc2VsZi5hY3Qoc2VsZi5jaG9tcDEoc2VsZi5jb252MSh4KSkpKQogICAgICAgIG91dCA9IHNlbGYuZHJvcG91dChzZWxmLmFjdChzZWxmLmNob21wMihzZWxmLmNvbnYyKG91dCkpKSkKICAgICAgICByZXMgPSB4IGlmIHNlbGYuZG93bnNhbXBsZSBpcyBOb25lIGVsc2Ugc2VsZi5kb3duc2FtcGxlKHgpCiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KG91dCArIHJlcykKCgpjbGFzcyBUQ05TaWduTW9kZWwobm4uTW9kdWxlKToKICAgICIiIlN0YWNrZWQgZGlsYXRlZCBUQ04gYmxvY2tzIC0+IGdsb2JhbCBhdmVyYWdlIHBvb2wgLT4gRkMgY2xhc3NpZmllci4iIiIKCiAgICB0YXJnZXRfZnJhbWVzID0gVEFSR0VUX0ZSQU1FUwoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0X2RpbTogaW50ID0gMTgwLAogICAgICAgIG51bV9jbGFzc2VzOiBpbnQgPSAxMCwKICAgICAgICBjaGFubmVsczogaW50ID0gQ0hBTk5FTFMsCiAgICAgICAga2VybmVsX3NpemU6IGludCA9IEtFUk5FTF9TSVpFLAogICAgICAgIGRpbGF0aW9uczogdHVwbGVbaW50LCAuLi5dID0gRElMQVRJT05TLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VULAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuaW5wdXRfZGltID0gaW50KGlucHV0X2RpbSkKICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gaW50KG51bV9jbGFzc2VzKQoKICAgICAgICBibG9ja3M6IGxpc3Rbbm4uTW9kdWxlXSA9IFtdCiAgICAgICAgaW5fY2ggPSBzZWxmLmlucHV0X2RpbQogICAgICAgIGZvciBkaWxhdGlvbiBpbiBkaWxhdGlvbnM6CiAgICAgICAgICAgIGJsb2Nrcy5hcHBlbmQoX1RlbXBvcmFsQmxvY2soaW5fY2gsIGNoYW5uZWxzLCBrZXJuZWxfc2l6ZSwgZGlsYXRpb24sIGRyb3BvdXQpKQogICAgICAgICAgICBpbl9jaCA9IGNoYW5uZWxzCiAgICAgICAgc2VsZi5uZXR3b3JrID0gbm4uU2VxdWVudGlhbCgqYmxvY2tzKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dCkpCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uTGluZWFyKGNoYW5uZWxzLCBzZWxmLm51bV9jbGFzc2VzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICMgKEIsIFQsIEQpIC0+IChCLCBELCBUKSBmb3IgQ29udjFkIG92ZXIgdGhlIHRpbWUgYXhpcy4KICAgICAgICB4ID0geC50cmFuc3Bvc2UoMSwgMikKICAgICAgICB4ID0gc2VsZi5uZXR3b3JrKHgpCiAgICAgICAgcG9vbGVkID0gdG9yY2gubWVhbih4LCBkaW09MikgICAgICMgZ2xvYmFsIGF2ZXJhZ2UgcG9vbCBvdmVyIHRpbWUKICAgICAgICBwb29sZWQgPSBzZWxmLmRyb3BvdXQocG9vbGVkKQogICAgICAgIHJldHVybiBzZWxmLmNsYXNzaWZpZXIocG9vbGVkKQoKCmRlZiBidWlsZF9tb2RlbChpbnB1dF9kaW06IGludCwgbnVtX2NsYXNzZXM6IGludCkgLT4gVENOU2lnbk1vZGVsOgogICAgcmV0dXJuIFRDTlNpZ25Nb2RlbChpbnB1dF9kaW09aW5wdXRfZGltLCBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcykK', 'transformer_sign.py': 'IiIiQ29tcGFjdCBUcmFuc2Zvcm1lciBlbmNvZGVyIGZvciBCSVNJTkRPIHNpZ24gY2xhc3NpZmljYXRpb24uCgpJbnB1dCBwcm9qZWN0aW9uIHRvIGQ9MTI4ICsgbGVhcm5lZCBwb3NpdGlvbmFsIGVtYmVkZGluZ3MsIHR3byBwcmUtbm9ybQpUcmFuc2Zvcm1lckVuY29kZXIgbGF5ZXJzICg0IGhlYWRzLCBGRiAyNTYpLCB0aGVuIG1lYW4gcG9vbGluZyBvdmVyIHRpbWUKKH4wLjMwTSBwYXJhbXMgYXQgMTgwLUQpLiBTZWxmLWF0dGVudGlvbiBjYXB0dXJlcyBsb25nLXJhbmdlIGZyYW1lIHJlbGF0aW9ucwp0aGF0IGEgc2luZ2xlIEdSVSBwYXNzIGNhbiBtaXNzLgoKTW9kdWxlIGNvbnRyYWN0IChzaGFyZWQgd2l0aCBncnVfYWRpIGV0Yy4pOiBleHBvc2VzIFRBUkdFVF9GUkFNRVMsCkRFRkFVTFRfTFIvQkFUQ0hfU0laRS9FUE9DSFMvUEFUSUVOQ0UgYW5kIGBgYnVpbGRfbW9kZWwoaW5wdXRfZGltLCBudW1fY2xhc3NlcylgYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpUQVJHRVRfRlJBTUVTID0gNjAKREVGQVVMVF9MUiA9IDVlLTQKREVGQVVMVF9CQVRDSF9TSVpFID0gMzIKREVGQVVMVF9FUE9DSFMgPSAzMDAKREVGQVVMVF9EUk9QT1VUID0gMC4yMApERUZBVUxUX1BBVElFTkNFID0gNDAKCkRfTU9ERUwgPSAxMjgKTl9IRUFEID0gNApGRl9ESU0gPSAyNTYKTl9MQVlFUlMgPSAyCgoKY2xhc3MgVHJhbnNmb3JtZXJTaWduTW9kZWwobm4uTW9kdWxlKToKICAgICIiIkxpbmVhciBwcm9qICsgbGVhcm5lZCBwb3MtZW1iIC0+IFRyYW5zZm9ybWVyRW5jb2RlciB4MiAtPiBtZWFuIHBvb2wgLT4gRkMuIiIiCgogICAgdGFyZ2V0X2ZyYW1lcyA9IFRBUkdFVF9GUkFNRVMKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9kaW06IGludCA9IDE4MCwKICAgICAgICBudW1fY2xhc3NlczogaW50ID0gMTAsCiAgICAgICAgZF9tb2RlbDogaW50ID0gRF9NT0RFTCwKICAgICAgICBuX2hlYWQ6IGludCA9IE5fSEVBRCwKICAgICAgICBmZl9kaW06IGludCA9IEZGX0RJTSwKICAgICAgICBuX2xheWVyczogaW50ID0gTl9MQVlFUlMsCiAgICAgICAgbWF4X2xlbjogaW50ID0gVEFSR0VUX0ZSQU1FUywKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IERFRkFVTFRfRFJPUE9VVCwKICAgICkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmlucHV0X2RpbSA9IGludChpbnB1dF9kaW0pCiAgICAgICAgc2VsZi5udW1fY2xhc3NlcyA9IGludChudW1fY2xhc3NlcykKICAgICAgICBzZWxmLm1heF9sZW4gPSBpbnQobWF4X2xlbikKCiAgICAgICAgc2VsZi5pbnB1dF9wcm9qID0gbm4uTGluZWFyKHNlbGYuaW5wdXRfZGltLCBkX21vZGVsKQogICAgICAgIHNlbGYucG9zX2VtYmVkZGluZyA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcygxLCBzZWxmLm1heF9sZW4sIGRfbW9kZWwpKQogICAgICAgIG5uLmluaXQudHJ1bmNfbm9ybWFsXyhzZWxmLnBvc19lbWJlZGRpbmcsIHN0ZD0wLjAyKQogICAgICAgIGVuY29kZXJfbGF5ZXIgPSBubi5UcmFuc2Zvcm1lckVuY29kZXJMYXllcigKICAgICAgICAgICAgZF9tb2RlbD1kX21vZGVsLAogICAgICAgICAgICBuaGVhZD1uX2hlYWQsCiAgICAgICAgICAgIGRpbV9mZWVkZm9yd2FyZD1mZl9kaW0sCiAgICAgICAgICAgIGRyb3BvdXQ9ZmxvYXQoZHJvcG91dCksCiAgICAgICAgICAgIGJhdGNoX2ZpcnN0PVRydWUsCiAgICAgICAgICAgIG5vcm1fZmlyc3Q9VHJ1ZSwKICAgICAgICAgICAgYWN0aXZhdGlvbj0iZ2VsdSIsCiAgICAgICAgKQogICAgICAgIHNlbGYuZW5jb2RlciA9IG5uLlRyYW5zZm9ybWVyRW5jb2RlcihlbmNvZGVyX2xheWVyLCBudW1fbGF5ZXJzPW5fbGF5ZXJzLCBlbmFibGVfbmVzdGVkX3RlbnNvcj1GYWxzZSkKICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0oZF9tb2RlbCkKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KGZsb2F0KGRyb3BvdXQpKQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcihkX21vZGVsLCBzZWxmLm51bV9jbGFzc2VzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHN0ZXBzID0geC5zaGFwZVsxXQogICAgICAgIHggPSBzZWxmLmlucHV0X3Byb2ooeCkKICAgICAgICB4ID0geCArIHNlbGYucG9zX2VtYmVkZGluZ1s6LCA6c3RlcHMsIDpdCiAgICAgICAgeCA9IHNlbGYuZW5jb2Rlcih4KQogICAgICAgIHggPSBzZWxmLm5vcm0oeCkKICAgICAgICBwb29sZWQgPSB0b3JjaC5tZWFuKHgsIGRpbT0xKSAgICAgIyBtZWFuIHBvb2wgb3ZlciB0aW1lCiAgICAgICAgcG9vbGVkID0gc2VsZi5kcm9wb3V0KHBvb2xlZCkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHBvb2xlZCkKCgpkZWYgYnVpbGRfbW9kZWwoaW5wdXRfZGltOiBpbnQsIG51bV9jbGFzc2VzOiBpbnQpIC0+IFRyYW5zZm9ybWVyU2lnbk1vZGVsOgogICAgcmV0dXJuIFRyYW5zZm9ybWVyU2lnbk1vZGVsKGlucHV0X2RpbT1pbnB1dF9kaW0sIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQo=', 'gru_manager.py': '...', 'gru_experts.py': '...', 'jetson_runtime.py': 'IiIiSmV0c29uIENVREEgMTIuNiBkaWFnbm9zdGljcyBhbmQgbGl2ZS1kZXZpY2UgcG9saWN5IGZvciBHUlUgcnVudGltZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBpbXBvcnRsaWIKaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKRVhQRUNURURfSkVUU09OX0NVREEgPSAiMTIuNiIKQ1VEQTEyNl9FTlZfTkFNRSA9ICJlbnZfYmlzaW5kb19jdWRhMTI2IgoKCmRlZiBfaW1wb3J0X3RvcmNoKHRvcmNoX21vZHVsZTogQW55IHwgTm9uZSA9IE5vbmUpIC0+IHR1cGxlW0FueSB8IE5vbmUsIHN0cl06CiAgICBpZiB0b3JjaF9tb2R1bGUgaXMgbm90IE5vbmU6CiAgICAgICAgcmV0dXJuIHRvcmNoX21vZHVsZSwgIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2ggICMgdHlwZTogaWdub3JlCgogICAgICAgIHJldHVybiB0b3JjaCwgIiIKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJldHVybiBOb25lLCBzdHIoZXhjKQoKCmRlZiBfbm9ybWFsaXplX3ZhcmlhbnQodmFyaWFudDogc3RyKSAtPiBzdHI6CiAgICB2YWx1ZSA9IHN0cih2YXJpYW50IG9yICJhZGkiKS5zdHJpcCgpLmxvd2VyKCkucmVwbGFjZSgiLSIsICJfIikKICAgIGlmIHZhbHVlLnN0YXJ0c3dpdGgoImdydV8iKToKICAgICAgICB2YWx1ZSA9IHZhbHVlWzQ6XQogICAgc3VmZml4ID0gIl9kZW5nYW5fYXVnbWVudGFzaSIKICAgIGlmIHZhbHVlLmVuZHN3aXRoKHN1ZmZpeCk6CiAgICAgICAgdmFsdWUgPSB2YWx1ZVs6IC1sZW4oc3VmZml4KV0KICAgIGlmIHZhbHVlIGluIHsiYXV0byIsICJiZXN0In06CiAgICAgICAgcmV0dXJuICJhZGkiCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgaXNfamV0c29uX3BsYXRmb3JtKCkgLT4gYm9vbDoKICAgIGlmIG9zLmVudmlyb24uZ2V0KCJCSVNJTkRPX0ZPUkNFX0pFVFNPTiIpID09ICIxIjoKICAgICAgICByZXR1cm4gVHJ1ZQogICAgbWFya2VycyA9ICgKICAgICAgICBQYXRoKCIvZXRjL252X3RlZ3JhX3JlbGVhc2UiKSwKICAgICAgICBQYXRoKCIvc3lzL21vZHVsZS90ZWdyYV9mdXNlIiksCiAgICAgICAgUGF0aCgiL3Vzci9saWIvYWFyY2g2NC1saW51eC1nbnUvdGVncmEiKSwKICAgICkKICAgIGlmIGFueShwYXRoLmV4aXN0cygpIGZvciBwYXRoIGluIG1hcmtlcnMpOgogICAgICAgIHJldHVybiBUcnVlCiAgICBtYWNoaW5lID0gcGxhdGZvcm0ubWFjaGluZSgpLmxvd2VyKCkKICAgIHJlbGVhc2UgPSBwbGF0Zm9ybS51bmFtZSgpLnJlbGVhc2UubG93ZXIoKQogICAgcmV0dXJuIG1hY2hpbmUgaW4geyJhYXJjaDY0IiwgImFybTY0In0gYW5kICJ0ZWdyYSIgaW4gcmVsZWFzZQoKCmRlZiB0b3JjaF9jdWRhX3ZlcnNpb24odG9yY2hfbW9kdWxlOiBBbnkgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgdG9yY2gsIF8gPSBfaW1wb3J0X3RvcmNoKHRvcmNoX21vZHVsZSkKICAgIGlmIHRvcmNoIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuICIiCiAgICB2YWx1ZSA9IGdldGF0dHIoZ2V0YXR0cih0b3JjaCwgInZlcnNpb24iLCBOb25lKSwgImN1ZGEiLCAiIikKICAgIGlmIHZhbHVlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuICIiCiAgICByZXR1cm4gc3RyKHZhbHVlKQoKCmRlZiB0b3JjaF9jdWRhX2F2YWlsYWJsZSh0b3JjaF9tb2R1bGU6IEFueSB8IE5vbmUgPSBOb25lKSAtPiBib29sOgogICAgdG9yY2gsIF8gPSBfaW1wb3J0X3RvcmNoKHRvcmNoX21vZHVsZSkKICAgIGlmIHRvcmNoIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGJvb2wodG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIGN1ZGFfdmVyc2lvbl9tYXRjaGVzKHRvcmNoX21vZHVsZTogQW55IHwgTm9uZSA9IE5vbmUsIGV4cGVjdGVkOiBzdHIgPSBFWFBFQ1RFRF9KRVRTT05fQ1VEQSkgLT4gYm9vbDoKICAgIHZlcnNpb24gPSB0b3JjaF9jdWRhX3ZlcnNpb24odG9yY2hfbW9kdWxlKQogICAgcmV0dXJuIG5vdCB2ZXJzaW9uIG9yIHZlcnNpb24uc3RhcnRzd2l0aChzdHIoZXhwZWN0ZWQpKQoKCmRlZiBjdWRhX2d1YXJkX21lc3NhZ2UodG9yY2hfbW9kdWxlOiBBbnkgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgdmVyc2lvbiA9IHRvcmNoX2N1ZGFfdmVyc2lvbih0b3JjaF9tb2R1bGUpIG9yICJDUFUtb25seS91bmtub3duIgogICAgZXhlY3V0YWJsZSA9IHN5cy5leGVjdXRhYmxlCiAgICByZXR1cm4gKAogICAgICAgIGYiUHlUb3JjaCBDVURBIGFrdGlmIGFkYWxhaCB7dmVyc2lvbn0sIHNlbWVudGFyYSBKZXRQYWNrIDYuMi4yIGJ1dHVoIENVREEge0VYUEVDVEVEX0pFVFNPTl9DVURBfS4gIgogICAgICAgIGYiUGFrYWkge0NVREExMjZfRU5WX05BTUV9OiAuL3NjcmlwdHMvc2V0dXBfamV0c29uX2N1ZGExMjYuc2ggbGFsdSBqYWxhbmthbiAiCiAgICAgICAgZiIuL3NjcmlwdHMvcnVuX2xpdmVfamV0c29uX2dydS5zaC4gUHl0aG9uIHNla2FyYW5nOiB7ZXhlY3V0YWJsZX0iCiAgICApCgoKZGVmIHZhbGlkYXRlX2N1ZGExMjZfZm9yX2pldHNvbigKICAgIHRvcmNoX21vZHVsZTogQW55IHwgTm9uZSA9IE5vbmUsCiAgICByZXF1aXJlX2N1ZGE6IGJvb2wgPSBGYWxzZSwKKSAtPiB0dXBsZVtib29sLCBzdHJdOgogICAgdG9yY2gsIGltcG9ydF9lcnJvciA9IF9pbXBvcnRfdG9yY2godG9yY2hfbW9kdWxlKQogICAgaWYgdG9yY2ggaXMgTm9uZToKICAgICAgICByZXR1cm4gRmFsc2UsIGYiUHlUb3JjaCB0aWRhayBiaXNhIGRpaW1wb3J0OiB7aW1wb3J0X2Vycm9yfSIKCiAgICBjdWRhX3ZlcnNpb24gPSB0b3JjaF9jdWRhX3ZlcnNpb24odG9yY2gpCiAgICBpZiBjdWRhX3ZlcnNpb24gYW5kIG5vdCBjdWRhX3ZlcnNpb24uc3RhcnRzd2l0aChFWFBFQ1RFRF9KRVRTT05fQ1VEQSk6CiAgICAgICAgcmV0dXJuIEZhbHNlLCBjdWRhX2d1YXJkX21lc3NhZ2UodG9yY2gpCgogICAgY3VkYV9hdmFpbGFibGUgPSB0b3JjaF9jdWRhX2F2YWlsYWJsZSh0b3JjaCkKICAgIGlmIHJlcXVpcmVfY3VkYSBhbmQgbm90IGN1ZGFfYXZhaWxhYmxlOgogICAgICAgIHJldHVybiBGYWxzZSwgKAogICAgICAgICAgICBmIkNVREEgdGlkYWsgdGVyc2VkaWEgZGFyaSBQeVRvcmNoIGluaS4gUGFzdGlrYW4gbWVuamFsYW5rYW4ge0NVREExMjZfRU5WX05BTUV9ICIKICAgICAgICAgICAgZiJkYW4gdG9yY2ggQ1VEQSB7RVhQRUNURURfSkVUU09OX0NVREF9LiIKICAgICAgICApCgogICAgaWYgY3VkYV92ZXJzaW9uOgogICAgICAgIHJldHVybiBUcnVlLCBmInRvcmNoIENVREEge2N1ZGFfdmVyc2lvbn0sIGN1ZGFfYXZhaWxhYmxlPXtjdWRhX2F2YWlsYWJsZX0iCiAgICByZXR1cm4gVHJ1ZSwgIlB5VG9yY2ggQ1BVLW9ubHk7IENVREEgdGlkYWsgZGltaW50YS4iCgoKZGVmIHNlbGVjdF9saXZlX2RldmljZSgKICAgIHZhcmlhbnQ6IHN0ciwKICAgIHJlcXVlc3RlZDogc3RyID0gImF1dG8iLAogICAgdG9yY2hfbW9kdWxlOiBBbnkgfCBOb25lID0gTm9uZSwKKSAtPiB0dXBsZVtzdHIsIHN0cl06CiAgICAiIiJQaWNrIHRoZSBmYXN0ZXN0IHNhZmUgbGl2ZSBkZXZpY2UgZm9yIGEgR1JVIHZhcmlhbnQgb24gSmV0c29uLgoKICAgIFRoZSBwb2xpY3kgaXMgYmFzZWQgb24gbG9jYWwgSmV0c29uIGJlbmNobWFyayBvYnNlcnZhdGlvbnM6CiAgICBBZGkgaXMgbW9yZSBhY2N1cmF0ZSBidXQgc2xvd2VyIG9uIENVREEgYmVjYXVzZSBpdHMgY3VzdG9tIFJlTFUtR1JVIGxvb3BzCiAgICBpbiBQeXRob247IEtodWt1aCBiZW5lZml0cyBmcm9tIENVREE7IEh5YnJpZCBkZWZhdWx0cyBDUFUgdW50aWwgdHJhaW5lZCBhbmQKICAgIGJlbmNobWFya2VkLgogICAgIiIiCgogICAgcmVxdWVzdGVkID0gc3RyKHJlcXVlc3RlZCBvciAiYXV0byIpLnN0cmlwKCkubG93ZXIoKQogICAgdmFyaWFudF9uYW1lID0gX25vcm1hbGl6ZV92YXJpYW50KHZhcmlhbnQpCgogICAgaWYgcmVxdWVzdGVkID09ICJjcHUiOgogICAgICAgIG9rLCBtc2cgPSB2YWxpZGF0ZV9jdWRhMTI2X2Zvcl9qZXRzb24odG9yY2hfbW9kdWxlPXRvcmNoX21vZHVsZSwgcmVxdWlyZV9jdWRhPUZhbHNlKQogICAgICAgIGlmIG9rOgogICAgICAgICAgICByZXR1cm4gImNwdSIsICJyZXF1ZXN0ZWRfY3B1IgogICAgICAgIHJldHVybiAiY3B1IiwgZiJyZXF1ZXN0ZWRfY3B1OyB3YXJuaW5nOiB7bXNnfSIKCiAgICBpZiByZXF1ZXN0ZWQgPT0gImN1ZGEiOgogICAgICAgIG9rLCBtc2cgPSB2YWxpZGF0ZV9jdWRhMTI2X2Zvcl9qZXRzb24odG9yY2hfbW9kdWxlPXRvcmNoX21vZHVsZSwgcmVxdWlyZV9jdWRhPVRydWUpCiAgICAgICAgaWYgbm90IG9rOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IobXNnKQogICAgICAgIHJldHVybiAiY3VkYSIsICJyZXF1ZXN0ZWRfY3VkYV92YWxpZF9jdWRhMTI2IgoKICAgIGlmIHJlcXVlc3RlZCAhPSAiYXV0byI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGV2aWNlIGhhcnVzIHNhbGFoIHNhdHU6IGF1dG8sIGNwdSwgY3VkYSIpCgogICAgb2ssIG1zZyA9IHZhbGlkYXRlX2N1ZGExMjZfZm9yX2pldHNvbih0b3JjaF9tb2R1bGU9dG9yY2hfbW9kdWxlLCByZXF1aXJlX2N1ZGE9RmFsc2UpCiAgICBpZiBub3Qgb2s6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKG1zZykKCiAgICBpZiB2YXJpYW50X25hbWUgPT0gImFkaSI6CiAgICAgICAgcmV0dXJuICJjcHUiLCAicG9saWN5X2FkaV9jcHVfZmFzdGVyX3RoYW5fY3VkYV9vbl9qZXRzb24iCgogICAgaWYgdmFyaWFudF9uYW1lID09ICJraHVrdWgiOgogICAgICAgIGlmIHRvcmNoX2N1ZGFfYXZhaWxhYmxlKHRvcmNoX21vZHVsZSk6CiAgICAgICAgICAgIHJldHVybiAiY3VkYSIsICJwb2xpY3lfa2h1a3VoX2N1ZGFfZmFzdGVyX29uX2pldHNvbiIKICAgICAgICByZXR1cm4gImNwdSIsIGYicG9saWN5X2todWt1aF9jcHVfZmFsbGJhY2s7IHttc2d9IgoKICAgIGlmIHZhcmlhbnRfbmFtZSA9PSAiaHlicmlkIjoKICAgICAgICByZXR1cm4gImNwdSIsICJwb2xpY3lfaHlicmlkX2NwdV91bnRpbF9sb2NhbF9iZW5jaG1hcmtfcHJlZmVyc19jdWRhIgoKICAgIHJldHVybiAiY3B1IiwgZiJwb2xpY3lfdW5rbm93bl92YXJpYW50X2NwdV9mYWxsYmFjazp7dmFyaWFudF9uYW1lfSIKCgpkZWYgX3RlbnNvcnJ0X3N0YXR1cygpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgdHJ5OgogICAgICAgIHRydCA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKCJ0ZW5zb3JydCIpCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIm9rIjogVHJ1ZSwKICAgICAgICAgICAgInZlcnNpb24iOiBzdHIoZ2V0YXR0cih0cnQsICJfX3ZlcnNpb25fXyIsICIiKSksCiAgICAgICAgICAgICJlcnJvciI6ICIiLAogICAgICAgIH0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJldHVybiB7Im9rIjogRmFsc2UsICJ2ZXJzaW9uIjogIiIsICJlcnJvciI6IHN0cihleGMpfQoKCmRlZiBfb3BlbmN2X2dzdHJlYW1lcl9zdGF0dXMoY3YyX21vZHVsZTogQW55IHwgTm9uZSkgLT4gc3RyOgogICAgaWYgY3YyX21vZHVsZSBpcyBOb25lOgogICAgICAgIHRyeToKICAgICAgICAgICAgY3YyX21vZHVsZSA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKCJjdjIiKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJldHVybiAidW5rbm93biIKICAgIHRyeToKICAgICAgICBidWlsZCA9IGN2Ml9tb2R1bGUuZ2V0QnVpbGRJbmZvcm1hdGlvbigpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiAidW5rbm93biIKICAgIGZvciBsaW5lIGluIHN0cihidWlsZCkuc3BsaXRsaW5lcygpOgogICAgICAgIGlmICJHU3RyZWFtZXI6IiBpbiBsaW5lOgogICAgICAgICAgICByZXR1cm4gbGluZS5zcGxpdCgiR1N0cmVhbWVyOiIsIDEpWzFdLnN0cmlwKCkKICAgIHJldHVybiAidW5rbm93biIKCgpkZWYgZGlhZ25vc3RpY3ModG9yY2hfbW9kdWxlOiBBbnkgfCBOb25lID0gTm9uZSwgY3YyX21vZHVsZTogQW55IHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgdG9yY2gsIGltcG9ydF9lcnJvciA9IF9pbXBvcnRfdG9yY2godG9yY2hfbW9kdWxlKQogICAgdHJ0ID0gX3RlbnNvcnJ0X3N0YXR1cygpCiAgICBkaWFnOiBkaWN0W3N0ciwgQW55XSA9IHsKICAgICAgICAicHl0aG9uIjogc3lzLmV4ZWN1dGFibGUsCiAgICAgICAgInN5c19wcmVmaXgiOiBzeXMucHJlZml4LAogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgIm1hY2hpbmUiOiBwbGF0Zm9ybS5tYWNoaW5lKCksCiAgICAgICAgImlzX2pldHNvbiI6IGlzX2pldHNvbl9wbGF0Zm9ybSgpLAogICAgICAgICJjdWRhX2hvbWUiOiBvcy5lbnZpcm9uLmdldCgiQ1VEQV9IT01FIiwgIiIpLAogICAgICAgICJ0b3JjaF9pbXBvcnRfb2siOiB0b3JjaCBpcyBub3QgTm9uZSwKICAgICAgICAidG9yY2hfaW1wb3J0X2Vycm9yIjogaW1wb3J0X2Vycm9yLAogICAgICAgICJ0b3JjaF92ZXJzaW9uIjogIiIsCiAgICAgICAgInRvcmNoX2N1ZGEiOiAiIiwKICAgICAgICAiY3VkYV9hdmFpbGFibGUiOiBGYWxzZSwKICAgICAgICAiZGV2aWNlX25hbWUiOiAiIiwKICAgICAgICAiZGV2aWNlX2NhcGFiaWxpdHkiOiAiIiwKICAgICAgICAidGVuc29ycnRfaW1wb3J0X29rIjogYm9vbCh0cnRbIm9rIl0pLAogICAgICAgICJ0ZW5zb3JydF92ZXJzaW9uIjogdHJ0WyJ2ZXJzaW9uIl0sCiAgICAgICAgInRlbnNvcnJ0X2Vycm9yIjogdHJ0WyJlcnJvciJdLAogICAgICAgICJvcGVuY3ZfdmVyc2lvbiI6ICIiLAogICAgICAgICJvcGVuY3ZfZ3N0cmVhbWVyIjogX29wZW5jdl9nc3RyZWFtZXJfc3RhdHVzKGN2Ml9tb2R1bGUpLAogICAgICAgICJ3YXJuaW5nIjogIiIsCiAgICB9CgogICAgaWYgY3YyX21vZHVsZSBpcyBOb25lOgogICAgICAgIHRyeToKICAgICAgICAgICAgY3YyX21vZHVsZSA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKCJjdjIiKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGN2Ml9tb2R1bGUgPSBOb25lCiAgICBpZiBjdjJfbW9kdWxlIGlzIG5vdCBOb25lOgogICAgICAgIGRpYWdbIm9wZW5jdl92ZXJzaW9uIl0gPSBzdHIoZ2V0YXR0cihjdjJfbW9kdWxlLCAiX192ZXJzaW9uX18iLCAiIikpCgogICAgaWYgdG9yY2ggaXMgTm9uZToKICAgICAgICBkaWFnWyJ3YXJuaW5nIl0gPSBmIlB5VG9yY2ggaW1wb3J0IGdhZ2FsOiB7aW1wb3J0X2Vycm9yfSIKICAgICAgICByZXR1cm4gZGlhZwoKICAgIHRyeToKICAgICAgICBkaWFnWyJ0b3JjaF92ZXJzaW9uIl0gPSBzdHIoZ2V0YXR0cih0b3JjaCwgIl9fdmVyc2lvbl9fIiwgIiIpKQogICAgICAgIGRpYWdbInRvcmNoX2N1ZGEiXSA9IHRvcmNoX2N1ZGFfdmVyc2lvbih0b3JjaCkKICAgICAgICBkaWFnWyJjdWRhX2F2YWlsYWJsZSJdID0gdG9yY2hfY3VkYV9hdmFpbGFibGUodG9yY2gpCiAgICAgICAgaWYgZGlhZ1siY3VkYV9hdmFpbGFibGUiXToKICAgICAgICAgICAgZGlhZ1siZGV2aWNlX25hbWUiXSA9IHN0cih0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgZGlhZ1siZGV2aWNlX2NhcGFiaWxpdHkiXSA9ICIuIi5qb2luKG1hcChzdHIsIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApKSkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIGRpYWdbImRldmljZV9jYXBhYmlsaXR5Il0gPSAiIgogICAgICAgIG9rLCBtc2cgPSB2YWxpZGF0ZV9jdWRhMTI2X2Zvcl9qZXRzb24odG9yY2gsIHJlcXVpcmVfY3VkYT1GYWxzZSkKICAgICAgICBpZiBub3Qgb2s6CiAgICAgICAgICAgIGRpYWdbIndhcm5pbmciXSA9IG1zZwogICAgICAgIGVsaWYgbm90IGRpYWdbImN1ZGFfYXZhaWxhYmxlIl06CiAgICAgICAgICAgIGRpYWdbIndhcm5pbmciXSA9ICJDVURBIHRpZGFrIHRlcnNlZGlhOyBsaXZlIEdSVSBha2FuIG1lbWFrYWkgQ1BVLiIKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGRpYWdbIndhcm5pbmciXSA9IHN0cihleGMpCiAgICByZXR1cm4gZGlhZwoKCmRlZiBkaWFnbm9zdGljc190ZXh0KGRpYWc6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZSA9IE5vbmUpIC0+IHN0cjoKICAgIGRhdGEgPSBkaWFnbm9zdGljcygpIGlmIGRpYWcgaXMgTm9uZSBlbHNlIGRpYWcKICAgIGxpbmVzID0gWwogICAgICAgIGYicHl0aG9uOiB7ZGF0YS5nZXQoJ3B5dGhvbicsICctJyl9IiwKICAgICAgICBmInRvcmNoOiB7ZGF0YS5nZXQoJ3RvcmNoX3ZlcnNpb24nLCAnLScpfSBjdWRhPXtkYXRhLmdldCgndG9yY2hfY3VkYScsICctJyl9IiwKICAgICAgICBmImN1ZGFfYXZhaWxhYmxlOiB7ZGF0YS5nZXQoJ2N1ZGFfYXZhaWxhYmxlJywgRmFsc2UpfSIsCiAgICAgICAgZiJkZXZpY2U6IHtkYXRhLmdldCgnZGV2aWNlX25hbWUnLCAnLScpIG9yICctJ30gY2M9e2RhdGEuZ2V0KCdkZXZpY2VfY2FwYWJpbGl0eScsICctJykgb3IgJy0nfSIsCiAgICAgICAgZiJ0ZW5zb3JydDoge2RhdGEuZ2V0KCd0ZW5zb3JydF9pbXBvcnRfb2snLCBGYWxzZSl9IHtkYXRhLmdldCgndGVuc29ycnRfdmVyc2lvbicsICcnKX0iLAogICAgICAgIGYib3BlbmN2OiB7ZGF0YS5nZXQoJ29wZW5jdl92ZXJzaW9uJywgJy0nKSBvciAnLSd9IGdzdHJlYW1lcj17ZGF0YS5nZXQoJ29wZW5jdl9nc3RyZWFtZXInLCAnLScpfSIsCiAgICBdCiAgICB3YXJuaW5nID0gZGF0YS5nZXQoIndhcm5pbmciKQogICAgaWYgd2FybmluZzoKICAgICAgICBsaW5lcy5hcHBlbmQoZiJ3YXJuaW5nOiB7d2FybmluZ30iKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykK'}
for rel, b64 in EMBEDDED.items():
    path = os.path.join(SRC_ROOT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        f.write(base64.b64decode(b64))
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)
print("Source modul v3 siap di", SRC_ROOT)
print("Files:", sorted(EMBEDDED.keys()))


In [ ]:
#@title 4. Konfigurasi (centang model / schema / mode / suite) { display-mode: "form" }
import feature_schemas as fs
import gru_manager as gm
import gru_experts as ge

#@markdown ### Model dasar (boleh banyak)
USE_KHUKUH = False  #@param {type:"boolean"}
USE_ADI = True  #@param {type:"boolean"}
USE_HYBRID = False  #@param {type:"boolean"}
USE_BIATTN = False  #@param {type:"boolean"}
USE_CONVFRONT = False  #@param {type:"boolean"}
USE_TCN = False  #@param {type:"boolean"}
USE_TRANSFORMER = False  #@param {type:"boolean"}

#@markdown ### Schema dataset (boleh banyak)
USE_SMART180 = True  #@param {type:"boolean"}
USE_KHUKUH1629 = False  #@param {type:"boolean"}
USE_ADI1662 = False  #@param {type:"boolean"}
USE_SMART180_FACE1584 = False  #@param {type:"boolean"}
USE_SMART180_MOUTHDYN214 = False  #@param {type:"boolean"}
USE_SMART180_MOUTHSTAT206 = False  #@param {type:"boolean"}
USE_SMART180_HANDFACE220 = False  #@param {type:"boolean"}
USE_SMART180_HANDFACE_VEL286 = False  #@param {type:"boolean"}

#@markdown ### Mode data
USE_ORIGINAL = True  #@param {type:"boolean"}
USE_AUGMENTED = False  #@param {type:"boolean"}

#@markdown ### Suite (main = model dasar; chunk10/threshold = expert per grup; boosted = stacker)
SUITE_MAIN = True  #@param {type:"boolean"}
SUITE_CHUNK10 = False  #@param {type:"boolean"}
SUITE_THRESHOLD = False  #@param {type:"boolean"}
SUITE_BOOSTED = False  #@param {type:"boolean"}

#@markdown ### Override hyperparameter per model (default mengikuti repo; boleh diedit)
KHUKUH_LR = 0.0001  #@param {type:"number"}
KHUKUH_PATIENCE = 25  #@param {type:"integer"}
KHUKUH_EPOCHS = 100  #@param {type:"integer"}
KHUKUH_BATCH = 64  #@param {type:"integer"}
ADI_LR = 0.001  #@param {type:"number"}
ADI_PATIENCE = 40  #@param {type:"integer"}
ADI_EPOCHS = 300  #@param {type:"integer"}
ADI_BATCH = 32  #@param {type:"integer"}
HYBRID_LR = 0.0001  #@param {type:"number"}
HYBRID_PATIENCE = 30  #@param {type:"integer"}
HYBRID_EPOCHS = 150  #@param {type:"integer"}
HYBRID_BATCH = 64  #@param {type:"integer"}
BIATTN_LR = 0.001  #@param {type:"number"}
BIATTN_PATIENCE = 40  #@param {type:"integer"}
BIATTN_EPOCHS = 300  #@param {type:"integer"}
BIATTN_BATCH = 32  #@param {type:"integer"}
CONVFRONT_LR = 0.001  #@param {type:"number"}
CONVFRONT_PATIENCE = 40  #@param {type:"integer"}
CONVFRONT_EPOCHS = 300  #@param {type:"integer"}
CONVFRONT_BATCH = 32  #@param {type:"integer"}
TCN_LR = 0.001  #@param {type:"number"}
TCN_PATIENCE = 40  #@param {type:"integer"}
TCN_EPOCHS = 300  #@param {type:"integer"}
TCN_BATCH = 32  #@param {type:"integer"}
TRANSFORMER_LR = 0.0005  #@param {type:"number"}
TRANSFORMER_PATIENCE = 40  #@param {type:"integer"}
TRANSFORMER_EPOCHS = 300  #@param {type:"integer"}
TRANSFORMER_BATCH = 32  #@param {type:"integer"}

#@markdown ### Suite expert (epoch terpisah supaya chunk10/threshold tidak terlalu panjang)
SUITE_EPOCHS = 60  #@param {type:"integer"}
THRESHOLD_DISTANCE = 0.0  #@param {type:"number"}
PRETRAIN_MAIN_FOR_EXPERTS = True  #@param {type:"boolean"}

#@markdown ### Global
DEVICE = "auto"  #@param ["auto", "cuda", "cpu"]
COPY_TO_LOCAL = True  #@param {type:"boolean"}
OVERWRITE_EXISTING = False  #@param {type:"boolean"}
LIMIT_PER_CLASS = 0  #@param {type:"integer"}
DRIVE_DATASET_DIR = "/content/drive/MyDrive/dataset_parquets"  #@param {type:"string"}
DRIVE_MODEL_DIR = "/content/drive/MyDrive/bisindo_models"  #@param {type:"string"}

MODEL_FLAGS = [
    ("khukuh", USE_KHUKUH),
    ("adi", USE_ADI),
    ("hybrid", USE_HYBRID),
    ("biattn", USE_BIATTN),
    ("convfront", USE_CONVFRONT),
    ("tcn", USE_TCN),
    ("transformer", USE_TRANSFORMER),
]
SCHEMA_FLAGS = [
    ("smart180", USE_SMART180),
    ("khukuh1629", USE_KHUKUH1629),
    ("adi1662", USE_ADI1662),
    ("smart180_face1584", USE_SMART180_FACE1584),
    ("smart180_mouthdyn214", USE_SMART180_MOUTHDYN214),
    ("smart180_mouthstat206", USE_SMART180_MOUTHSTAT206),
    ("smart180_handface220", USE_SMART180_HANDFACE220),
    ("smart180_handface_vel286", USE_SMART180_HANDFACE_VEL286),
]
SUITE_FLAGS = [
    ("main", SUITE_MAIN),
    ("chunk10", SUITE_CHUNK10),
    ("threshold", SUITE_THRESHOLD),
    ("boosted", SUITE_BOOSTED),
]

BASES = [name for name, on in MODEL_FLAGS if on]
SCHEMAS = [name for name, on in SCHEMA_FLAGS if on]
MODES = (["original"] if USE_ORIGINAL else []) + (["with_augmentation"] if USE_AUGMENTED else [])
SUITES = [name for name, on in SUITE_FLAGS if on]
EXPERT_SUITES = [name for name in SUITES if name != "main"]

PARAMS = {
    "khukuh": {"lr": KHUKUH_LR, "patience": KHUKUH_PATIENCE, "epochs": KHUKUH_EPOCHS, "batch": KHUKUH_BATCH},
    "adi": {"lr": ADI_LR, "patience": ADI_PATIENCE, "epochs": ADI_EPOCHS, "batch": ADI_BATCH},
    "hybrid": {"lr": HYBRID_LR, "patience": HYBRID_PATIENCE, "epochs": HYBRID_EPOCHS, "batch": HYBRID_BATCH},
    "biattn": {"lr": BIATTN_LR, "patience": BIATTN_PATIENCE, "epochs": BIATTN_EPOCHS, "batch": BIATTN_BATCH},
    "convfront": {"lr": CONVFRONT_LR, "patience": CONVFRONT_PATIENCE, "epochs": CONVFRONT_EPOCHS, "batch": CONVFRONT_BATCH},
    "tcn": {"lr": TCN_LR, "patience": TCN_PATIENCE, "epochs": TCN_EPOCHS, "batch": TCN_BATCH},
    "transformer": {"lr": TRANSFORMER_LR, "patience": TRANSFORMER_PATIENCE, "epochs": TRANSFORMER_EPOCHS, "batch": TRANSFORMER_BATCH},
}

assert BASES, "Pilih minimal 1 model."
assert SCHEMAS, "Pilih minimal 1 schema."
assert MODES, "Pilih minimal 1 mode data (original / with_augmentation)."
assert SUITES, "Pilih minimal 1 suite."

unknown_models = [name for name in BASES if name not in gm.BASE_VARIANT_NAMES]
unknown_schemas = [name for name in SCHEMAS if name not in fs.SCHEMA_NAMES]
unknown_suites = [name for name in SUITES if name not in ge.SUITE_CHOICES]
assert not unknown_models, f"Model tidak dikenal repo: {unknown_models}"
assert not unknown_schemas, f"Schema tidak dikenal repo: {unknown_schemas}"
assert not unknown_suites, f"Suite tidak dikenal repo: {unknown_suites}"

print("Model dasar:", BASES)
print("Schema     :", SCHEMAS)
print("Mode data  :", MODES)
print("Suite      :", SUITES)
print("Overwrite  :", bool(OVERWRITE_EXISTING))
print("Total kombinasi model x schema x mode x suite:", len(BASES) * len(SCHEMAS) * len(MODES) * len(SUITES))


In [ ]:
#@title 5. Siapkan dataset (copy schema terpilih: Drive -> disk lokal)
import os, shutil, glob, time

if COPY_TO_LOCAL:
    DATASET_DIR = "/content/dataset_parquets"
else:
    DATASET_DIR = DRIVE_DATASET_DIR

for schema in SCHEMAS:
    src_dir = os.path.join(DRIVE_DATASET_DIR, schema)
    assert os.path.isdir(src_dir), f"Folder schema tidak ada di Drive: {src_dir}"
    files = sorted(glob.glob(os.path.join(src_dir, "*.parquet")))
    assert files, f"Tidak ada *.parquet di {src_dir}"
    if COPY_TO_LOCAL:
        dst_dir = os.path.join(DATASET_DIR, schema)
        os.makedirs(dst_dir, exist_ok=True)
        t0 = time.time()
        copied = 0
        for fp in files:
            dst = os.path.join(dst_dir, os.path.basename(fp))
            if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(fp):
                shutil.copy2(fp, dst)
                copied += 1
        print(f"[{schema}] {len(files)} parquet tersedia, {copied} disalin dalam {time.time()-t0:.1f}s -> {dst_dir}")
    else:
        print(f"[{schema}] pakai langsung dari Drive ({len(files)} parquet)")

print("DATASET_DIR =", DATASET_DIR)


In [ ]:
#@title 6. Training (main + suite terpilih) — memakai gru_experts.train_suite
import torch
import gru_manager as gm
import gru_experts as ge

print("Device:", "cuda" if torch.cuda.is_available() else "cpu",
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only"))

MODEL_DIR_LOCAL = "/content/models"   # ROOT_DIR/models lokal, fresh tiap sesi Colab
BACKUP_DIR_LOCAL = "/content/backups"
limit = (int(LIMIT_PER_CLASS) or None)
thr = None if float(THRESHOLD_DISTANCE) == 0.0 else float(THRESHOLD_DISTANCE)
log = []

for schema in SCHEMAS:
    for base in BASES:
        P = PARAMS[base]
        for mode in MODES:
            variant = gm.normalize_variant_name(base)
            if mode == "with_augmentation":
                variant = gm.augmented_variant_name(variant)
            tag = f"{schema}/{variant}"
            print("
" + "=" * 72)
            print("KOMBINASI:", tag, "| suite:", SUITES)
            print("=" * 72)

            main_kwargs = dict(
                variant=variant,
                schema=schema,
                dataset_dir=DATASET_DIR,
                model_dir=MODEL_DIR_LOCAL,
                epochs=int(P["epochs"]),
                batch_size=int(P["batch"]),
                lr=float(P["lr"]),
                patience=int(P["patience"]),
                device=DEVICE,
                threshold=thr,
                limit_per_class=limit,
                overwrite_existing=bool(OVERWRITE_EXISTING),
                backup_root=BACKUP_DIR_LOCAL,
                train_data=gm.variant_train_data_mode(variant),
            )
            suite_kwargs = dict(main_kwargs)
            suite_kwargs["epochs"] = int(SUITE_EPOCHS)

            try:
                if "main" in SUITES:
                    result = ge.train_suite(suites="main", **main_kwargs)
                    ok = bool(result.get("main", {}).get("ok", True)) if isinstance(result, dict) else True
                    log.append((tag, "main", ok))

                if EXPERT_SUITES:
                    if PRETRAIN_MAIN_FOR_EXPERTS and not gm.checkpoint_exists(variant, MODEL_DIR_LOCAL, schema=schema):
                        print("[pretrain main] checkpoint main belum ada; train main dengan epoch default model.")
                        result = ge.train_suite(suites="main", **main_kwargs)
                        ok = bool(result.get("main", {}).get("ok", True)) if isinstance(result, dict) else True
                        log.append((tag, "main-pretrain", ok))

                    result = ge.train_suite(suites=",".join(EXPERT_SUITES), **suite_kwargs)
                    if isinstance(result, dict):
                        for suite_name in EXPERT_SUITES:
                            log.append((tag, suite_name, suite_name in result))
                    else:
                        for suite_name in EXPERT_SUITES:
                            log.append((tag, suite_name, True))
            except Exception as exc:
                print(f"[ERROR] {tag}: {exc}")
                log.append((tag, "+".join(SUITES), False))
                raise

print("
Ringkasan:")
for tag, suite, ok in log:
    print(f"  [{'OK' if ok else 'GAGAL'}] {tag} :: {suite}")


In [ ]:
#@title 7. Simpan semua checkpoint (termasuk experts/) ke Google Drive + ringkasan
import os, shutil, glob, json

for schema in SCHEMAS:
    local_dir = os.path.join("/content/models", "gru", schema)
    if not os.path.isdir(local_dir):
        print("(lewati, belum ada)", local_dir)
        continue
    drive_dir = os.path.join(DRIVE_MODEL_DIR, "gru", schema)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    n = sum(len(fs) for _, _, fs in os.walk(local_dir))
    print(f"[{schema}] {n} file -> {drive_dir}")

    for fp in sorted(glob.glob(os.path.join(local_dir, "*_metadata.json"))):
        meta = json.load(open(fp))
        print("  main:", os.path.basename(fp),
              "| classes=", meta.get("num_classes"),
              "| target_frames=", meta.get("target_frames"),
              "| best_val_acc=", round(float(meta.get("best_val_acc", 0)), 4),
              "| epochs_run=", meta.get("epochs_run"),
              "| data=", meta.get("training_data_mode"))

    for fp in sorted(glob.glob(os.path.join(local_dir, "experts", "*", "*", "suite_metadata.json"))):
        meta = json.load(open(fp))
        groups = meta.get("groups") or []
        print("  suite:", meta.get("suite"), os.path.relpath(os.path.dirname(fp), local_dir),
              "| groups=", len(groups), "| threshold=", meta.get("threshold"))


## Selesai

Semua artefak tersimpan di `DRIVE_MODEL_DIR/gru/<schema>/`:
- **main**: `gru_<variant>.pth`, `_labels.json`, `_metadata.json`
- **expert**: `experts/<suite>/gru_<variant>/chunk_NNN/gru_expert.*`, `prototypes.npz`, `suite_metadata.json`
- **boosted**: `experts/boosted/gru_<variant>/boosted_stack.pkl` (+ prototypes + metadata)

Pakai di repo Jetson: copy folder `gru/<schema>/` itu ke `models/gru/<schema>/`. Suite/route seperti `main_chunk10`, `vote_all`, dan `boosted_stack` otomatis kebaca saat evaluasi/live karena struktur folder sama dengan `gru_experts.RoutedGRUPredictor`.

Tips: mulai dari default kecil dulu (`adi + smart180 + main`). Setelah itu baru centang schema/model/suite tambahan supaya durasi training tetap kebaca.
